# **Body Performance Analytics & Intelligent Classification System**

---

### **Project Metadata**
- **Objective:** End-to-end analysis and predictive modeling of physiological performance metrics.
- **Scope:** Data Cleaning, Exploratory Data Analysis (EDA), Classification, and Regression.
- **Status:** Final Revision for Submission

## **Snipers Team:**
- **Gharieb (Leader)**
- **Saad**
- **El Kholy**
- **Wael**
- **Hesham**

## **0. Project Overview**

---

This notebook represents a comprehensive merge of the team's research and development efforts. Our methodology adheres to the standard Machine Learning Pipeline as outlined in the Course Handbook:

1.  **Data Understanding & Cleaning:** Initial profiling and rectification of physiological anomalies.
2.  **Exploratory Data Analysis (EDA):** Feature distribution analysis and correlation mapping.
3.  **Predictive Modeling:** Training of robust classification (Performance Grade) and regression (Broad Jump) models.
4.  **Model Validation:** Employment of K-Fold Cross-Validation and multi-split testing.
5.  **Interactive Deployment:** Implementation of a Gradio interface for real-time inference.

## **1. Data Understanding & Initial Cleaning**

---

In this phase, we focus on establishing a clean baseline for the model. This involves auditing the dataset structure, verifying logical constraints (such as age limits and blood pressure physics), and implementing an initial cleaning strategy to handle outliers and duplicate records.

### 1.1 Imports and Global Settings

In [ ]:
# ===============================
# 1.1 Imports and Global Settings
# ===============================

# Install missing dependencies if needed
!pip install -q gradio

# Core libraries
import os
import math
import requests
import gdown
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Gradio
import gradio as gr

# Scikit-learn: Preprocessing and Model Selection
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

# Scikit-learn: Metrics
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score
)

# Scikit-learn: Classification Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier

# Visualization Extras
from matplotlib.patches import Patch

# Configure Visualization Styles
sns.set(style="whitegrid", context="notebook")
sns.set_theme(style="whitegrid", palette="muted")

### 1.2 Raw Dataset Loading & First Look

In [ ]:
# ==================================================================
# 1.2 Data Acquisition: Loading and Initial Inspection
# ==================================================================

# Dataset source configuration
google_drive_url = "https://drive.google.com/file/d/1eW88Bgg-tBv5eYn915Vp2ltkoLXkV6zT/view?usp=sharing"
file_id = google_drive_url.split('/')[-2]
path = "/content/bodyPerformance.csv"

# Secure download handling
if not os.path.exists(path):
    print(f"[INFO] Downloading dataset from secure cloud storage...")
    try:
        gdown.download(id=file_id, output=path, quiet=False)
    except Exception as e:
        raise FileNotFoundError(f"[ERROR] Acquisition failed: {e}")
else:
    print(f"[INFO] Dataset detected locally at {path}. Pre-loading sequence initiated.")

# DataFrame initialization
df = pd.read_csv(path)

print(f"\n[SUCCESS] Dataset loaded. Dimensionality: {df.shape[0]} rows x {df.shape[1]} columns")
display(df.head())

In [ ]:
# This cell checked the validity of the previous GitHub URL. It is no longer needed.

### **1.3 Column Definitions, Constraints and Business Meaning**
This table explains the significance of each feature and the logical constraints applied to the data.

| Column Name | Meaning | Data Type | Constraints |
| :--- | :--- | :--- | :--- |
| **age** | Age of the individual | Numerical (int) | 20 to 65 (Standard range) |
| **gender** | Biological sex | Categorical (M/F) | Only 'M' or 'F' |
| **height_cm** | Height in centimeters | Numerical (float) | Must be positive (> 0) |
| **weight_kg** | Weight in kilograms | Numerical (float) | Must be positive (> 0) |
| **body fat_%** | Percentage of body fat | Numerical (float) | Range: 0 to 100 |
| **diastolic** | Blood pressure (min) | Numerical (float) | Positive value |
| **systolic** | Blood pressure (max) | Numerical (float) | Positive value |
| **gripForce** | Strength of hand grip | Numerical (float) | Positive value |
| **sit and bend forward_cm** | Flexibility test (cm) | Numerical (float) | Can be positive/negative |
| **sit-ups counts** | Abdominal strength | Numerical (int) | Non-negative integers |
| **broad jump_cm** | Leg power / Jumping distance | Numerical (float) | Positive value |
| **class** | Fitness Grade (Target) | Categorical (A,B,C,D) | 'A' (Best) to 'D' (Worst) |

### 1.4 Data Profiling and Statistical Summary
In this step, we inspect the data types, check for missing values, and look at the statistical distribution of the features.

**df.info()**: To check column data types and verify that there are no initial null values.
**df.describe()**: To get a statistical overview (mean, min, max, etc.) and detect any immediate outliers or illogical values (e.g., zero height or negative values).

---
The following output verifies that the actual data types in our DataFrame match the expected types defined in the table above.

In [ ]:
# ===============================
# 2.2 Basic numeric summary
# ===============================

df.describe()

In [ ]:
#---Displaying general information about the dataset---
df.info()

### 1.5 Data Types, Missing Values and Initial Profiling

In [ ]:
# ===============================
# 1.5.1 Data types overview
# ===============================

df.dtypes.to_frame("dtype")

In [ ]:
# ===============================
# 1.5.2 Target class distribution
# ===============================

class_counts = df["class"].value_counts()
class_pct = (class_counts / len(df) * 100).round(2)

display(pd.DataFrame({
    "count": class_counts,
    "percentage": class_pct
}))

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="class", order=sorted(df["class"].unique()))
plt.title("Class Distribution (A/B/C/D)")
plt.xlabel("Performance Class")
plt.ylabel("Count")
plt.show()

### 1.6 Data Cleaning Rules and Implementation

In [ ]:
# ===============================
# 1.6.1 Missing values analysis
# ===============================
# Check for null values in each column
null_counts = df.isnull().sum()

# Display columns that have missing values (if any)
print("Missing values per column:")
print(null_counts)

# Total number of missing values in the whole dataset
print(f"\nTotal missing values: {null_counts.sum()}")

## **2. Data Quality & Statistical Analysis**

---

### **2.1 Duplicate Detection and Handling**
Duplicate records can skew statistical distributions and lead to data leakage between training and testing sets. We identify and remove exact row duplicates to ensure unique observations.

### 2.1 Duplicate Detection and Handling

Duplicate data can occur during data collection or entry. Removing them is crucial because they can:
1. Biased statistical: calculations like Mean and Standard Deviation.
2. Biased model: The ML model sees the same row twice during training — it learns that row is "more important" than others
3. Inflated accuracy: If a duplicate appears in both training and testing data, the model already "knows" that row — giving falsely high accuracy                                                      
**Action:** We will identify the number of duplicates and remove them, keeping only the first occurrence.

In [ ]:
# ==================================================================
# 2.1.1 Redundancy Audit
# ==================================================================

duplicate_count = df.duplicated().sum()
print(f"[INFO] Duplicate rows detected: {duplicate_count}")

if duplicate_count > 0:
    df.drop_duplicates(inplace=True)
    print(f"[SUCCESS] Redundancy removed. New shape: {df.shape}")
else:
    print("[INFO] No duplicates found. Data integrity maintained.")

In [ ]:
# Optionally inspect duplicates if any
if duplicate_count > 0:
    display(df[df.duplicated()].head())

In [ ]:
# Show the actual duplicate rows
# keep=False means show ALL copies (both the original and the duplicate)
duplicates = df[df.duplicated(keep=False)]
print(f"Rows involved in duplication: {len(duplicates)}")
print(duplicates)

In [ ]:
# Remove duplicates (if any)
if duplicate_count > 0:
    df.drop_duplicates(inplace=True)
    print("Duplicates removed successfully!")
    print(f"New dataset shape: {df.shape}")
else:
    print("No duplicates found. Dataset is clean from repetition.")

### **2.2 Physiological Validity Audit**
We apply domain-specific constraints to verify that the physiological data is realistic (e.g., Systolic > Diastolic).

## 📌 Valid Ranges for Every Column

Before writing code you need to know what "valid" means for each column:

| Column | Valid Range | Why |
|---|---|---|
| `age` | 18 – 80 | Participants are adults in fitness evaluation |
| `height_cm` | 100 – 220 | Realistic human height range |
| `weight_kg` | 20 – 250 | Must be positive; extreme obesity upper bound |
| `body fat_%` | 3 – 65 | Below 3% is dangerous; above 65% is unrealistic |
| `diastolic` | 40 – 130 | Medical normal and hypertensive range |
| `systolic` | 70 – 200 | Medical normal and hypertensive range |
| `systolic > diastolic` | Always true | Physiological law — cannot be violated |
| `gripForce` | 0 – 100 | Non-negative; world record ~116 kgf |
| `sit and bend forward_cm` | -30 – 60 | Negative is valid (poor flexibility) |
| `sit-ups counts` | 0 – 100 | Non-negative whole numbers |
| `broad jump_cm` | 0 – 350 | Non-negative; world record ~340 cm |
| `gender` | M or F only | Only two valid categories |
| `class` | A, B, C, D only | Only four valid performance labels |

In [ ]:
# 1. Ensure dataset is loaded
path = "/content/bodyPerformance.csv"
if os.path.exists(path):
    df = pd.read_csv(path)
    # 2. Re-apply duplicate removal to ensure clean state
    df.drop_duplicates(inplace=True)
    print(f"Dataset loaded and duplicates removed. Shape: {df.shape}")

    # 3. BP Logic Check
    invalid_bp_logic = df[df['systolic'] <= df['diastolic']]
    print(f"Systolic <= diastolic: {len(invalid_bp_logic)} violation(s)")
    if len(invalid_bp_logic) > 0:
        display(invalid_bp_logic[['systolic', 'diastolic']].head())
else:
    print("Dataset file not found at /content/bodyPerformance.csv")

In [ ]:
# Displaying statistical summary of numeric columns rounded to 2 decimal places
display(df.describe().round(2))

# Displaying general information about the dataset
print("\n--- Dataset Information ---")
df.info()

In [ ]:
# Filtering rows where age is less than 18 or greater than 80
invalid_age_rows = df[(df['age'] < 18) | (df['age'] > 80)]

print(f"Found {len(invalid_age_rows)} rows with age violations (outside 18-80 range):")
if len(invalid_age_rows) > 0:
    display(invalid_age_rows)
else:
    print("All age values are within the valid range.")

In [ ]:
# Filtering rows where systolic pressure is less than or equal to diastolic
invalid_bp_rows = df[df['systolic'] <= df['diastolic']]

print(f"Found {len(invalid_bp_rows)} rows with blood pressure logic violations:")
display(invalid_bp_rows)

# Why systolic must always be greater than diastolic:

- Blood pressure is written as systolic / diastolic — for example 120/80
- Systolic = pressure when heart beats (higher pressure)
- Diastolic = pressure when heart rests (lower pressure)
- The heart always exerts more pressure when beating than when resting
- So systolic must always be larger — if it isn't, that measurement is physically impossible

In [ ]:
# Detailed Validity Range Checks
if 'df' in locals():
    results = {
        "Invalid height (<100 or >220)": len(df[(df['height_cm'] < 100) | (df['height_cm'] > 220)]),
        "Invalid body fat (<3 or >65)": len(df[(df['body fat_%'] < 3) | (df['body fat_%'] > 65)]),
        "Negative grip force": len(df[df['gripForce'] < 0]),
        "Invalid sit-ups (<0 or >100)": len(df[(df['sit-ups counts'] < 0) | (df['sit-ups counts'] > 100)]),
        "Negative broad jump": len(df[df['broad jump_cm'] < 0]),
        "Invalid gender labels": len(df[~df['gender'].isin(['M', 'F'])]),
        "Invalid class labels": len(df[~df['class'].isin(['A', 'B', 'C', 'D'])])
    }

    for check, count in results.items():
        print(f"{check:30}: {count} violation(s)")
else:
    print("DataFrame 'df' is still not defined. Please run the previous cell first.")

### 2.3 Validity Violations Report and Decisions
Before applying corrections, we generate a comprehensive audit to understand the extent of data corruption. This report maps each violation to a severity level and a corresponding corrective action (Capping or Removal).

In [ ]:
# ==================================================================
# 2.3.1 Comprehensive Validity Audit
# ==================================================================

def audit_dataset(dataframe):
    audit_log = []

    # 1. Blood Pressure Logic (Critical)
    bp_mask = dataframe['systolic'] <= dataframe['diastolic']
    audit_log.append({'Metric': 'BP Logic', 'Condition': 'Sys <= Dia', 'Violations': bp_mask.sum(), 'Action': 'Removal'})

    # 2. Physical Impossibilities (Critical)
    weight_mask = dataframe['weight_kg'] <= 0
    grip_mask = dataframe['gripForce'] < 0
    jump_mask = dataframe['broad jump_cm'] < 0
    audit_log.append({'Metric': 'Phys. Limits', 'Condition': 'Weight/Force < 0', 'Violations': weight_mask.sum() + grip_mask.sum() + jump_mask.sum(), 'Action': 'Removal'})

    # 3. Out-of-Range Constraints (High/Medium)
    age_mask = (dataframe['age'] < 18) | (dataframe['age'] > 80)
    audit_log.append({'Metric': 'Age Range', 'Condition': '<18 or >80', 'Violations': age_mask.sum(), 'Action': 'Capping'})

    return pd.DataFrame(audit_log)

report_df = audit_dataset(df)
print("--- DATA VALIDITY AUDIT REPORT ---")
display(report_df)

In [ ]:
# ==================================================================
# 2.3.2 Visualizing Audit Violations
# ==================================================================

# Convert report_df to a list of records for processing
report = report_df.to_dict('records')

# Extract violation counts
checks   = [r['Metric'] for r in report]
counts   = [r['Violations'] for r in report]
actions  = [r['Action'] for r in report]

# Color by action type
color_map = {'Removal': 'red', 'Capping': 'orange'}
colors = [color_map[a] for a in actions]

# Plot
plt.figure(figsize=(10, 6))
bars = plt.bar(checks, counts, color=colors, edgecolor='black')

# Add count labels on top of each bar
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.1,
             str(count),
             ha='center', fontsize=10, fontweight='bold')

plt.title('Data Quality Audit: Detected Violations', fontsize=14, fontweight='bold')
plt.xlabel('Metric Type')
plt.ylabel('Number of Violations')

# Legend
legend_elements = [Patch(facecolor='red', label='Critical (Removal)'),
                   Patch(facecolor='orange', label='Warning (Capping)')]
plt.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

In [ ]:
print("Applying violation handling actions...")
print("=" * 50)

# ── Action 1: Remove rows with BP logic violation ──────────────────────
bp_logic_mask = df['systolic'] <= df['diastolic']
bp_violations = bp_logic_mask.sum()
df = df[~bp_logic_mask].reset_index(drop=True)
print(f"✅ Removed {bp_violations} rows where systolic <= diastolic")

# ── Action 2: Cap diastolic blood pressure ─────────────────────────────
df['diastolic'] = df['diastolic'].clip(lower=40, upper=130)
print(f"✅ Diastolic capped to range [40, 130]")

# ── Action 3: Cap systolic blood pressure ──────────────────────────────
df['systolic'] = df['systolic'].clip(lower=70, upper=200)
print(f"✅ Systolic capped to range [70, 200]")

# ── Action 4: Remove negative weight (if any) ─────────────────────────
neg_weight = (df['weight_kg'] <= 0).sum()
df = df[df['weight_kg'] > 0].reset_index(drop=True)
print(f"✅ Removed {neg_weight} rows with non-positive weight")

# ── Action 5: Cap age to valid range ──────────────────────────────────
df['age'] = df['age'].clip(lower=18, upper=80)
print(f"✅ Age capped to range [18, 80]")

print()
print(f"Final dataset shape after all corrections: {df.shape}")

In [ ]:
# Run all checks again to confirm everything is clean
print("POST-CLEANING VERIFICATION")
print("=" * 50)

checks_after = {
    'weight_kg <= 0'        : (df['weight_kg'] <= 0).sum(),
    'age < 18 or > 80'      : ((df['age']<18)|(df['age']>80)).sum(),
    'diastolic out of range' : ((df['diastolic']<40)|(df['diastolic']>130)).sum(),
    'systolic out of range'  : ((df['systolic']<70)|(df['systolic']>200)).sum(),
    'systolic <= diastolic'  : (df['systolic'] <= df['diastolic']).sum(),
    'negative gripForce'     : (df['gripForce'] < 0).sum(),
    'negative broad jump'    : (df['broad jump_cm'] < 0).sum(),
}

for check, count in checks_after.items():
    status = "✅ CLEAN" if count == 0 else f"⚠️ {count} remaining"
    print(f"{check:35} : {status}")

print()
total_remaining = sum(checks_after.values())
if total_remaining == 0:
    print("✅ ALL VIOLATIONS RESOLVED — Dataset is valid and clean.")
else:
    print(f"⚠️ {total_remaining} violations still need attention.")

### 2.4 Apply Cleaning Rules and Create Clean Dataset
Following the audit, we execute the 'Clean-State' sequence. We favor capping for physiological outliers (preserving data) and removal for logical impossibilities.

In [ ]:
# ==================================================================
# 2.4.1 Execution of Cleaning Protocol
# ==================================================================

# 1. Logical Removal: Impossible BP and non-positive metrics
clean_df = df[df['systolic'] > df['diastolic']].copy()
clean_df = clean_df[(clean_df['weight_kg'] > 0) & (clean_df['gripForce'] >= 0) & (clean_df['broad jump_cm'] >= 0)]

# 2. Physiological Capping: Clamp values to medical realistic bounds
clean_df['diastolic'] = clean_df['diastolic'].clip(40, 130)
clean_df['systolic'] = clean_df['systolic'].clip(70, 200)
clean_df['age'] = clean_df['age'].clip(18, 80)

print(f"[CLEANING COMPLETE] Original: {df.shape[0]} | Cleaned: {clean_df.shape[0]}")

### **2.5 Descriptive Statistics for Numeric Features: Mean, Median, Std Dev, Min, Max & Skewness**

1️⃣ **Mean** (Average)
What it is: Add all values together, then divide by how many there are                                                              
2️⃣ **Median** (Middle Value)
What it is: Sort all values from smallest to largest — the median is the value exactly in the middle                                     
3️⃣ **Standard Deviation** (Std Dev)
What it is: Measures how spread out the values are around the mean    
4️⃣ **Minimum** (Min)
What it is: The smallest value in the column                        
5️⃣ **Maximum** (Max)
What it is: The largest value in the column                      
6️⃣ **Skewness**

What it is: Measures how asymmetric the distribution of values is    
Three possible situations:
1. Symmetric (bell curve): Values spread equally on both sides of mean
2. Right skewed: Most values are low, but a few very high values pull the tail to the right
3. Left skewed: Most values are high, but a few very low values pull the tail to the left

In [ ]:
# ====================================
# Descriptive statistics on clean data
# Get only numeric columns
# =====================================

numeric_cols = df.select_dtypes(include='number').columns

# Build complete statistics table
stats_table = pd.DataFrame({
    'Mean'      : df[numeric_cols].mean().round(2),
    'Median'    : df[numeric_cols].median().round(2),
    'Std Dev'   : df[numeric_cols].std().round(2),
    'Min'       : df[numeric_cols].min().round(2),
    '25%'       : df[numeric_cols].quantile(0.25).round(2),
    '50%'       : df[numeric_cols].quantile(0.50).round(2),
    '75%'       : df[numeric_cols].quantile(0.75).round(2),
    'Max'       : df[numeric_cols].max().round(2),
    'Skewness'  : df[numeric_cols].skew().round(3)
})

print("Complete Descriptive Statistics Table:")
stats_table

In [ ]:
# Add a column that interprets the skewness value automatically
def interpret_skewness(skew_val):
    if abs(skew_val) < 0.5:
        return 'Normal ✅'
    elif abs(skew_val) < 1.0:
        return 'Moderate skew ⚠️'
    else:
        return 'High skew ❌'

stats_table['Skew Type'] = stats_table['Skewness'].apply(interpret_skewness)
stats_table['Direction'] = stats_table['Skewness'].apply(
    lambda x: 'Right (+)' if x > 0.5
              else 'Left (-)' if x < -0.5
              else 'Symmetric'
)

print("Statistics with Skewness Interpretation:")
stats_table

### 2.6 Univariate Distribution Analysis (Skewness & Spread)
**Univariate** means one variable at a time. We analyze each column individually through histograms and boxplots to understand the shape of the distribution and identify remaining outliers.

**Univariate** means one variable at a time. You analyze each column individually — looking at its shape, spread, and behavior through both numbers and visualizations.
                                      
📌 Two Types of Features — Two Different Approaches                   
🔢 **Numeric Features** → Use Histogram + Boxplot

Histogram shows the shape of the distribution                         
Boxplot shows the spread and outliers

🔤 **Categorical Features** → Use Bar Chart (Count Plot)

Shows how many participants fall into each category                  
Reveals class balance or imbalance


In [ ]:
numeric_cols = df.select_dtypes(include='number').columns
skewness_values = df[numeric_cols].skew().round(3)

# Color bars by skewness severity
colors = []
for val in skewness_values:
    if abs(val) < 0.5:
        colors.append('green')      # normal
    elif abs(val) < 1.0:
        colors.append('orange')     # moderate
    else:
        colors.append('red')        # high skew

plt.figure(figsize=(12, 5))
bars = plt.bar(skewness_values.index,
               skewness_values.values,
               color=colors,
               edgecolor='black')

# Add value labels on bars
for bar, val in zip(bars, skewness_values.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.02,
             f'{val:.2f}',
             ha='center', fontsize=9)

# Reference lines
plt.axhline(y=0.5,  color='orange', linestyle='--',
            label='Moderate threshold (+0.5)')
plt.axhline(y=-0.5, color='orange', linestyle='--')
plt.axhline(y=1.0,  color='red',    linestyle='--',
            label='High skew threshold (+1.0)')
plt.axhline(y=-1.0, color='red',    linestyle='--')
plt.axhline(y=0,    color='black',  linestyle='-', linewidth=0.8)

plt.title('Skewness of Each Numeric Column', fontsize=14)
plt.xlabel('Column')
plt.ylabel('Skewness Value')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

What this chart shows:

- Green bars → normally distributed columns → safe to use as-is
- Orange bars → moderately skewed → use median for statistics
- Red bars → highly skewed → consider log transformation before ML
- The dashed lines show the thresholds at ±0.5 and ±1.0

In [ ]:
# ===============================
# 2.5.1 Extended Statistics with Skewness
# ===============================

numeric_cols = clean_df.select_dtypes(include=["float64", "int64"]).columns

stats_list = []
for col in numeric_cols:
    s = clean_df[col]
    stats_list.append({
        "Feature": col,
        "Mean": s.mean(),
        "Median": s.median(),
        "Std Dev": s.std(),
        "Min": s.min(),
        "Max": s.max(),
        "Skewness": s.skew()
    })

extended_stats_df = pd.DataFrame(stats_list).round(2)
display(extended_stats_df)

### 2.7 Statistical Insights and Data Quality Notes

**Key statistical observations**

- Most core body measurements (height, weight, body fat) are roughly symmetric with moderate spread, suggesting a diverse but well‑behaved sample.  
- Sit-ups and broad jump show slight right skew, indicating a subset of very high performers with above‑average results.
- Blood pressure values lie mostly within realistic adult ranges after cleaning, confirming that our validity rules were effective.


## **3. Data Visualization & Exploratory Data Analysis**

---

### **3.1 Bivariate Analysis: Metrics vs. Performance Class**
In this section, we analyze how key physical metrics correlate with the target variable (`class`). We use boxplots to compare the distribution of performance indicators across the four classes (A=Best, D=Worst).

### 3.1 Histograms for All Numeric Features

In [ ]:
# ===============================
# 2.6.1 Histograms for All Numeric Features
# ===============================

# Create a grid of histograms
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(clean_df[col], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].axvline(clean_df[col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean')
    axes[i].axvline(clean_df[col].median(), color='green', linestyle='--', linewidth=1.5, label=f'Median')
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].legend(fontsize=7)

# Hide empty subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Physiological and Performance Metrics', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.2 Boxplots and Outlier Exploration

In [ ]:
# ===============================
# Boxplots for numeric features
# ===============================


numeric_cols = df.select_dtypes(include=[np.number]).columns

num_cols = len(numeric_cols)
n_cols = 3
n_rows = math.ceil(num_cols / n_cols)

plt.figure(figsize=(5 * n_cols, 3.5 * n_rows))

for i, col in enumerate(numeric_cols, 1):
    ax = plt.subplot(n_rows, n_cols, i)

    # Boxplot
    sns.boxplot(
        x=df[col],
        color="#90caf9",
        linewidth=1.2,
        fliersize=2,
        ax=ax
    )

    # Outlier count using IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    n_outliers = len(outliers)

    ax.set_title(f"{col}", fontsize=11, fontweight="bold")
    ax.set_xlabel(f"Value  |  Outliers: {n_outliers}", fontsize=9)
    ax.grid(axis="x", alpha=0.3)

plt.suptitle("Boxplots of Numerical Features with Outlier Counts",
             y=1.02, fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Deep dive analysis — each column gets its own detailed plot


for col in numeric_cols:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    col_data = df[col].dropna()
    mean_val = col_data.mean()
    median_val = col_data.median()
    std_val = col_data.std()
    min_val = col_data.min()
    max_val = col_data.max()
    skew_val = col_data.skew()

    # ── Left: Histogram + KDE ─────────────────────────────────────
    sns.histplot(
        col_data,
        bins=30,
        kde=True,
        color="#1f77b4",
        edgecolor="black",
        ax=ax1
    )
    ax1.axvline(mean_val, color="red", linestyle="--", linewidth=1.5,
                label=f"Mean = {mean_val:.2f}")
    ax1.axvline(median_val, color="green", linestyle="-.", linewidth=1.5,
                label=f"Median = {median_val:.2f}")
    ax1.set_title(f"{col} – Distribution", fontweight="bold", fontsize=11)
    ax1.set_xlabel("Value", fontsize=9)
    ax1.set_ylabel("Frequency", fontsize=9)
    ax1.legend(fontsize=8)

    # ── Right: Horizontal Boxplot ────────────────────────────────
    sns.boxplot(
        x=col_data,
        color="#90caf9",
        linewidth=1.2,
        fliersize=3,
        ax=ax2
    )
    ax2.set_title(f"{col} – Boxplot", fontweight="bold", fontsize=11)
    ax2.set_xlabel("Value", fontsize=9)
    ax2.grid(axis="x", alpha=0.3)

    # ── Statistics panel ─────────────────────────────────────────
    stats_text = (
        f"Mean:    {mean_val:.2f}\n"
        f"Median:  {median_val:.2f}\n"
        f"Std Dev: {std_val:.2f}\n"
        f"Min:     {min_val:.2f}\n"
        f"Max:     {max_val:.2f}\n"
        f"Skew:    {skew_val:.3f}"
    )

    fig.text(
        0.98, 0.5, stats_text,
        fontsize=9,
        va="center",
        ha="right",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="wheat", alpha=0.6)
    )

    plt.suptitle(f"Univariate Analysis – {col}", fontsize=13, fontweight="bold", y=1.03)
    plt.tight_layout()
    plt.show()

----
## **Outlier Detection and Removal (IQR Method)**


---


Now that our data is numerical, we need to handle **Outliers** (extreme values that don't make sense).
- **Method:** We use the **Interquartile Range (IQR)** to define a "Normal Range".
- **Action:** Any value outside the range `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` will be considered an outlier and removed.
- **Why?** Outliers can mislead the model and decrease its accuracy.

**Mapping Reference:**
- Gender: 0 = Male, 1 = Female
- Class: 0=A, 1=B, 2=C, 3=D

In [ ]:
# Function to remove outliers using IQR
#def remove_outliers(df, columns):
    #for col in columns:
        #Q1 = df[col].quantile(0.25)
#        Q3 = df[col].quantile(0.75)
 #       IQR = Q3 - Q1
  #      lower_bound = Q1 - 1.5 * IQR
   #     upper_bound = Q3 + 1.5 * IQR

        # Filtering the data
    #    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    #return df

# List of numerical columns to check for outliers
#numerical_cols = ['age', 'height_cm', 'weight_kg', 'body fat_%', 'diastolic',
 #                 'systolic', 'gripForce', 'sit and bend forward_cm',
  #                'sit-ups counts', 'broad jump_cm']

# Shape before removal
#print(f"Shape before removing outliers: {df.shape}")

# Removing outliers
#df = remove_outliers(df, numerical_cols)

# Shape after removal
#print(f"Shape after removing outliers: {df.shape}")

### 3.3 Interpret Results & Discuss Distribution Characteristics

**Distribution and outlier insights**

- Height and weight distributions are close to normal; most participants cluster around average values with few extreme outliers.
- Sit and bend forward shows more dispersion and skewness, reflecting a wide range of flexibility among subjects.
- Boxplots confirm that outliers exist but generally correspond to plausible high‑performance individuals rather than pure noise


## 📊 Complete Distribution Interpretation Table

| Column | Distribution Shape | Key Statistical Observation | Real World Meaning | ML Implication |
|---|---|---|---|---|
| `age` | Right skewed | Mean (36.78) > Median (32.0) — skewed by older participants | Dataset dominated by young adults (20s-30s); fewer older participants | Consider log transform — skewness may affect distance-based models |
| `height_cm` | Approximately normal | Mean ≈ Median — balanced distribution around 168.6 cm | Heights follow natural human distribution — no anomalies | No transformation needed — good feature as-is |
| `weight_kg` | Right skewed | Mean (67.1) > Median (65.2) — few heavy participants raise average | Most participants have healthy weight; few overweight participants exist | Consider log transform — right skew present |
| `body fat_%` | Slightly right skewed | Mean slightly above median — mild skew from high fat values | Most participants have moderate body fat; few with very high fat % | Mild skew acceptable — monitor during modeling |
| `diastolic` | Approximately normal | Mean ≈ Median ≈ 80 mmHg — centered on normal BP range | Most participants have normal diastolic BP — healthy population | No transformation needed — normal distribution |
| `systolic` | Approximately normal | Mean ≈ Median ≈ 130 mmHg — centered on high-normal BP range | Many participants at high-normal systolic — fitness evaluation context | No transformation needed — normal distribution |
| `gripForce` | Approximately normal | Mean ≈ Median — grip strength evenly distributed | Grip strength varies but is fairly consistent across participants | No transformation needed — normal distribution |
| `sit and bend forward_cm` | Approximately normal | Near-symmetric — flexibility evenly spread around 13.5 cm | Flexibility varies widely — some very flexible, some very stiff | No transformation needed — normal distribution |
| `sit-ups counts` | Approximately normal | Near-symmetric — endurance evenly spread around 40 sit-ups | Sit-up performance fairly consistent — no extreme outlier groups | No transformation needed — normal distribution |
| `broad jump_cm` | Slightly left skewed | Mean (190) slightly below Median (193) — mild left skew | Most participants jump well — few very low jump distances exist | No transformation needed — near normal |

In [ ]:
# Visualize Each Distribution With Full Interpretation
# Detailed interpretation plots
numeric_cols = df.select_dtypes(include='number').columns.tolist()

for col in numeric_cols:

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ── Plot 1: Histogram with KDE curve ────────────────────────
    axes[0].hist(df[col], bins=30, color='steelblue',
                 edgecolor='black', alpha=0.6, density=True)

    # Add KDE (smooth curve showing distribution shape)
    df[col].plot.kde(ax=axes[0], color='darkblue', linewidth=2)

    axes[0].axvline(df[col].mean(),   color='red',
                    linestyle='--', linewidth=2,
                    label=f'Mean = {df[col].mean():.2f}')
    axes[0].axvline(df[col].median(), color='green',
                    linestyle='--', linewidth=2,
                    label=f'Median = {df[col].median():.2f}')

    axes[0].set_title(f'{col} — Histogram + KDE', fontweight='bold')
    axes[0].set_xlabel('Value')
    axes[0].set_ylabel('Density')
    axes[0].legend()

    # ── Plot 2: Boxplot ──────────────────────────────────────────
    axes[1].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='navy'),
                    medianprops=dict(color='red', linewidth=2.5),
                    whiskerprops=dict(color='navy', linewidth=1.5),
                    flierprops=dict(marker='o', markerfacecolor='red',
                                   markersize=4, alpha=0.4))

    # Calculate and show IQR
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1

    axes[1].set_title(f'{col} — Boxplot\nIQR = {IQR:.2f}',
                      fontweight='bold')
    axes[1].set_ylabel('Value')

    # ── Plot 3: Statistics Summary ───────────────────────────────
    axes[2].axis('off')  # turn off the axes — just show text

    skew_val = df[col].skew()

    # Determine skewness interpretation
    if abs(skew_val) < 0.5:
        skew_label = 'Approximately Normal ✅'
        skew_color = 'green'
    elif abs(skew_val) < 1.0:
        skew_label = 'Moderately Skewed ⚠️'
        skew_color = 'orange'
    else:
        skew_label = 'Highly Skewed ❌'
        skew_color = 'red'

    direction = 'Right (+)' if skew_val > 0.1 else \
                'Left (-)'  if skew_val < -0.1 else 'Symmetric'

    # Count outliers using IQR method
    outlier_count = len(df[
        (df[col] < Q1 - 1.5*IQR) |
        (df[col] > Q3 + 1.5*IQR)
    ])

    summary = (
        f"STATISTICS SUMMARY\n"
        f"{'─'*30}\n"
        f"Mean        : {df[col].mean():.3f}\n"
        f"Median      : {df[col].median():.3f}\n"
        f"Std Dev     : {df[col].std():.3f}\n"
        f"Min         : {df[col].min():.3f}\n"
        f"Max         : {df[col].max():.3f}\n"
        f"Q1 (25%)    : {Q1:.3f}\n"
        f"Q3 (75%)    : {Q3:.3f}\n"
        f"IQR         : {IQR:.3f}\n"
        f"Skewness    : {skew_val:.3f}\n"
        f"Direction   : {direction}\n"
        f"Shape       : {skew_label}\n"
        f"Outliers    : {outlier_count} rows\n"
        f"({'─'*30})\n"
        f"Mean vs Median:\n"
        f"{'Mean > Median → Right skew' if df[col].mean() > df[col].median() else 'Mean < Median → Left skew' if df[col].mean() < df[col].median() else 'Mean ≈ Median → Symmetric'}"
    )

    axes[2].text(0.05, 0.95, summary,
                transform=axes[2].transAxes,
                fontsize=10, verticalalignment='top',
                fontfamily='monospace',
                bbox=dict(boxstyle='round',
                         facecolor='lightyellow',
                         edgecolor='gray',
                         alpha=0.8))

    axes[2].set_title('Interpretation', fontweight='bold')

    plt.suptitle(f'Complete Univariate Analysis: {col}',
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print()

In [ ]:
# Categorical Distribution Interpretation
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Gender Distribution ──────────────────────────────────────────────
gender_counts = df['gender'].value_counts()
colors_gender = ['steelblue', 'coral']

wedges, texts, autotexts = axes[0].pie(
    gender_counts.values,
    labels=gender_counts.index,
    autopct='%1.1f%%',
    colors=colors_gender,
    startangle=90,
    explode=(0.05, 0.05)
)
axes[0].set_title('Gender Distribution', fontsize=13, fontweight='bold')

# Add count annotations
for i, (label, count) in enumerate(gender_counts.items()):
    autotexts[i].set_text(f'{count}\n({count/len(df)*100:.1f}%)')
    autotexts[i].set_fontsize(11)

# ── Class Distribution ───────────────────────────────────────────────
class_counts = df['class'].value_counts().sort_index()
colors_class = ['gold', 'mediumseagreen', 'steelblue', 'coral']

bars = axes[1].bar(class_counts.index,
                   class_counts.values,
                   color=colors_class,
                   edgecolor='black',
                   width=0.6)

# Add labels on bars
for bar, count in zip(bars, class_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 20,
                f'{count}\n({count/len(df)*100:.1f}%)',
                ha='center', fontsize=11, fontweight='bold')

axes[1].set_title('Performance Class Distribution',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Performance Class (A=Best, D=Lowest)')
axes[1].set_ylabel('Number of Participants')
axes[1].set_ylim(0, max(class_counts.values) * 1.2)

# Add reference line at perfect balance
perfect_balance = len(df) / 4
axes[1].axhline(y=perfect_balance,
                color='red', linestyle='--',
                label=f'Perfect balance = {perfect_balance:.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

## 📊 Complete Distribution Interpretation Report

---

### Age
The age distribution is **RIGHT SKEWED** (skewness ≈ +0.71).
Mean (36.78) is noticeably higher than Median (32.0).
Most participants are between 21–40 years old.
A smaller group of older participants (50–64) pulls the mean upward.
- → Dataset is dominated by younger adults
- → For ML: consider log transformation to reduce skewness impact

---

### Height
The height distribution is **APPROXIMATELY NORMAL** (skewness ≈ -0.35).
Mean (168.56) ≈ Median (169.2) — very close together.
Values range from 125 cm to 193.8 cm.
- → Natural human height follows a bell curve — this is expected
- → For ML: no transformation needed — already well distributed

---

### Weight
The weight distribution is **MODERATELY RIGHT SKEWED** (skewness ≈ +0.82).
Mean (67.1) > Median (65.2).
Most participants weigh between 50–85 kg.
A few heavier participants (>100 kg) create the right tail.
- → For ML: consider log transformation — skewness may affect models

---

### Body Fat %
**Slightly right skewed** (skewness ≈ +0.41) — near normal.
Mean (23.5%) is close to Median (22.8%).
Range: 3% to 52%.
- → Most participants have moderate body fat levels
- → The few very high values are likely overweight participants

---

### Diastolic Blood Pressure
**Approximately normal** distribution (skewness ≈ +0.15).
Mean ≈ Median ≈ 80 mmHg — centered on normal BP range.
Some outlier values exist at the high end (hypertensive patients).
- → Population is generally healthy in terms of diastolic BP

---

### Systolic Blood Pressure
**Approximately normal** (skewness ≈ +0.32).
Mean ≈ Median ≈ 130 mmHg — at the upper end of normal range.
- → Many participants at high-normal systolic — expected in a fitness evaluation study

---

### Grip Force
**Approximately normal** (skewness ≈ -0.02) — nearly perfect symmetry.
Mean ≈ Median ≈ 36.7 kgf.
- → Grip strength is evenly distributed across participants
- → No transformation needed — ideal distribution for ML

---

### Flexibility (Sit and Bend Forward)
**Approximately normal** (skewness ≈ -0.21).
Note: negative values are **VALID** — poor flexibility scores below zero.
Mean ≈ Median ≈ 13.5 cm.
- → Flexibility is fairly evenly distributed in this population

---

### Sit-ups Count
**Near normal** distribution (skewness ≈ -0.19).
Mean ≈ Median ≈ 40 sit-ups. Range: 0 to 80.
- → Muscular endurance is evenly spread — no extreme outlier groups

---

### Broad Jump
**Slightly left skewed** (skewness ≈ -0.44) — near normal.
Mean (190 cm) slightly below Median (193 cm).
- → Most participants jump well with a few very poor jumpers creating the left tail
- → Explosive power is the most evenly distributed performance metric

---

### Gender
Label Encode (A=0,B=1,C=2,D=3) — TARGET

---

### Class
Label Encode (M=0, F=1)

###3.3 Univariate Analysis (Distributions)
 This section expands on the histogram code you started, analyzing the distribution of each numerical feature.
 Step 12: Univariate Analysis (Feature Distributions)
 In this section, we visualize the distributions of our numerical features to understand their spread, identify central tendencies, and detect any  skewness. This is crucial for understanding the baseline of our physical performance metrics.

In [ ]:
# Set professional visual style
sns.set_theme(style="whitegrid", palette="muted")

# Select only numeric columns for histograms
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
numeric_cols = [col for col in numeric_cols if col not in ['gender', 'class']]

# Plotting Histograms
plt.figure(figsize=(18, 15))
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(4, 3, i)
    sns.histplot(df[col], kde=True, bins=30, color='royalblue')
    plt.title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    plt.xlabel('')
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

# For Univariate Analysis (Distributions)
Observation:
Normal Distributions: Physical attributes like `height_cm` and `weight_kg` follow a relatively normal (bell-shaped) distribution, indicating a balanced sample of body types.
Skewness: Variables like `age` show a slight right skew, meaning the majority of the individuals in this dataset are younger, with fewer participants in the older age brackets.
Performance Metrics: Metrics like `sit-ups counts` and `broad jump_cm` show wide variances, reflecting the diverse fitness levels present in the dataset before classifying them into performance tiers.

###3.4 Bivariate Analysis (Features vs. Performance Class)
This section answers how different physical metrics impact the final target class (A, B, C, D).
Step 13: Bivariate Analysis (Metrics vs. Target Class)
Here, we analyze how different physical metrics correlate with the target variable (`class`). We use boxplots to compare the distribution of key physical fitness indicators (like body fat, grip force, and jump distance) across the four performance classes.
Note: Class 'A' represents the highest level of performance.

In [ ]:
# ==================================================================
# 3.1.1 Boxplot Analysis: Performance Drivers
# ==================================================================

key_metrics = ['body fat_%', 'gripForce', 'sit-ups counts', 'broad jump_cm']
class_order = ['A', 'B', 'C', 'D']

plt.figure(figsize=(16, 10))
for i, col in enumerate(key_metrics, 1):
    plt.subplot(2, 2, i)
    sns.boxplot(x='class', y=col, data=clean_df, order=class_order, palette='viridis')
    plt.title(f'{col} across Performance Classes', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# For Bivariate Analysis (Features vs. Performance Class)
 Observation:
 Body Fat & Performance: There is a clear trend showing that individuals in Class 'A' (best performance) have significantly lower and more tightly clustered `body fat_%` compared to Class 'D'.
 Strength & Agility: Physical performance metrics (`gripForce`, `sit-ups counts`, and `broad jump_cm`) strictly decrease as the performance class drops from A to D. Class 'A' consistently displays the highest median scores and uppermost quartiles across all fitness tests.
 Outliers:Some outliers exist (e.g., individuals with high body fat still achieving Class B, or low grip force in Class A), indicating that the final classification is determined by a holistic combination of these metrics, not just one.

In [ ]:
# Define the numerical columns that have outliers
cols_with_outliers = ['body fat_%', 'gripForce', 'sit and bend forward_cm', 'broad jump_cm']

print("Applying IQR Capping to handle outliers...")

# Loop through each column to apply IQR Capping
for col in cols_with_outliers:
    # 1. Calculate Q1 (25th percentile) and Q3 (75th percentile)
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    # 2. Calculate the Interquartile Range (IQR)
    IQR = Q3 - Q1

    # 3. Define the upper and lower limits
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    # 4. Cap the outliers (Replace values outside the limits with the limit values)
    df[col] = df[col].clip(lower=lower_limit, upper=upper_limit)

print("Outliers have been successfully capped! The dataset is now cleaner without losing rows.")

## Categorical Analysis (Gender vs. Performance Class)
This section looks at the relationship between biological sex and fitness classifications.
Step 10 (§1.10): Categorical Analysis (Gender vs. Class)
In this section, we investigate the relationship between gender and performance class to see if the distribution of fitness levels varies significantly between males and females.

In [ ]:
plt.figure(figsize=(9, 6))
# Countplot to show class distribution separated by gender
sns.countplot(x='class', hue='gender', data=df, order=class_order, palette='viridis')

plt.title('Performance Class Distribution by Gender', fontsize=15, fontweight='bold')
plt.xlabel('Performance Class (A=Best, D=Worst)', fontsize=12)
plt.ylabel('Number of Individuals', fontsize=12)
plt.legend(title='Gender', loc='upper right')

plt.show()

# Scatter Plots: Feature vs. Feature by Class
To deepen our Bivariate Analysis, we use scatter plots to observe the relationship between two continuous physical metrics simultaneously, colored by our target variable (`class`). This helps us see if certain combinations of metrics create clear boundaries between performance levels.

In [ ]:
plt.figure(figsize=(14, 6))

# Scatter plot 1: Weight vs. Body Fat percentage
plt.subplot(1, 2, 1)
sns.scatterplot(x='weight_kg', y='body fat_%', hue='class', data=df,
                palette='coolwarm', hue_order=['A', 'B', 'C', 'D'], alpha=0.7)
plt.title('Weight vs. Body Fat % (Colored by Class)', fontsize=13, fontweight='bold')
plt.xlabel('Weight (kg)')
plt.ylabel('Body Fat (%)')

# Scatter plot 2: Broad Jump vs. Grip Force (Strength & Agility)
plt.subplot(1, 2, 2)
sns.scatterplot(x='broad jump_cm', y='gripForce', hue='class', data=df,
                palette='coolwarm', hue_order=['A', 'B', 'C', 'D'], alpha=0.7)
plt.title('Broad Jump vs. Grip Force (Colored by Class)', fontsize=13, fontweight='bold')
plt.xlabel('Broad Jump (cm)')
plt.ylabel('Grip Force')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 4))

plt.subplot(1, 3, 1)
sns.scatterplot(data=df, x="age", y="broad jump_cm", hue="class", alpha=0.6)
plt.title("Age vs Broad Jump by Class")

plt.subplot(1, 3, 2)
sns.scatterplot(data=df, x="body fat_%", y="broad jump_cm", hue="class", alpha=0.6)
plt.title("Body Fat % vs Broad Jump by Class")

plt.subplot(1, 3, 3)
sns.scatterplot(data=df, x="gripForce", y="broad jump_cm", hue="class", alpha=0.6)
plt.title("Grip Force vs Broad Jump by Class")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(18, 5))

common_kwargs = dict(
    data=df,
    hue="class",
    alpha=0.5,
    s=25,
    edgecolor="none"
)

# 1) Age vs Broad Jump
ax1 = plt.subplot(1, 3, 1)
sns.scatterplot(x="age", y="broad jump_cm", **common_kwargs)
ax1.set_title("Age vs Broad Jump by Class", fontsize=11, fontweight="bold")
ax1.set_xlabel("Age (years)", fontsize=9)
ax1.set_ylabel("Broad Jump (cm)", fontsize=9)

# 2) Body Fat vs Broad Jump
ax2 = plt.subplot(1, 3, 2)
sns.scatterplot(x="body fat_%", y="broad jump_cm", **common_kwargs)
ax2.set_title("Body Fat % vs Broad Jump by Class", fontsize=11, fontweight="bold")
ax2.set_xlabel("Body Fat %", fontsize=9)
ax2.set_ylabel("Broad Jump (cm)", fontsize=9)

# 3) Grip Force vs Broad Jump
ax3 = plt.subplot(1, 3, 3)
sns.scatterplot(x="gripForce", y="broad jump_cm", **common_kwargs)
ax3.set_title("Grip Force vs Broad Jump by Class", fontsize=11, fontweight="bold")
ax3.set_xlabel("Grip Force", fontsize=9)
ax3.set_ylabel("Broad Jump (cm)", fontsize=9)

# Shared legend
handles, labels = ax3.get_legend_handles_labels()
plt.legend(handles=handles, labels=labels, title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.suptitle("Key Relationships with Broad Jump by Class", y=1.05, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

Weight vs. Body Fat: We can see that individuals with lower body fat percentages tend to cluster toward the better performance classes (A and B), even if their overall weight varies. The 'D' class is heavily clustered at the higher end of the body fat spectrum.
Agility vs. Strength (Broad Jump vs. Grip Force):** There is a clear positive correlation here. As both broad jump distance and grip force increase, the density of Class 'A' and 'B' individuals heavily increases. Individuals scoring low on both metrics almost exclusively fall into Class 'C' or 'D'.

# Categorical Analysis (Gender vs. Performance Class)
Observation:
Class Distribution by Gender: Both genders are represented across all performance classes (A, B, C, D).
Proportions: The distribution of classes appears relatively balanced between males and females, though total counts for males are slightly higher in the dataset. This suggests that the grading/classification criteria for classes A-D likely scales appropriately according to biological sex standards rather than holding both genders to the exact same raw physical thresholds.

# Correlation Analysis (Heatmap)
This final step checks for multicollinearity (features that overlap in the information they provide) and strong predictors.

# Correlation Analysis
Finally, we compute and visualize the correlation matrix for all numerical features. This heatmap helps us identify:
1. Strong Predictors:Features highly correlated with performance.
2. Multicollinearity:Highly correlated predictors (e.g., weight and height) which might need to be handled before training certain machine learning models.

In [ ]:
# ==================================================================
# 3.2 Correlation Analysis: Multicollinearity Check
# ==================================================================

plt.figure(figsize=(12, 9))
corr_matrix = clean_df.select_dtypes(include=['float64', 'int64']).corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap of Physiological Features', fontsize=16, fontweight='bold')
plt.show()

In [ ]:
plt.figure(figsize=(11, 9))

corr = df[numeric_cols].corr()

# Optional: mask upper triangle to avoid duplication
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr,
    mask=mask,                 # comment this line if you want full matrix
    annot=True,                # show correlation values
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8, "label": "Correlation"},
    square=True
)

plt.title("Correlation Matrix of Numerical Features", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

# Observation:
Strong Positive Correlations: As expected, `weight_kg` and `height_cm` exhibit a high positive correlation (multicollinearity). Additionally, physical outputs like `broad jump_cm` and `gripForce` are strongly positively correlated with each other.
Strong Negative Correlations: `body fat_%` shows a strong negative correlation with most physical performance metrics (e.g., `sit-ups counts`, `broad jump_cm`). This indicates that higher body fat is a strong predictor of lower physical performance.
Blood Pressure: `systolic` and `diastolic` blood pressure are highly correlated with each other, but have very weak correlations with the physical fitness test results, implying they might not be the strongest standalone predictors for the final target class.

**Correlation insights**

- Grip force, sit‑ups, and broad jump are positively correlated, indicating that muscular strength, endurance, and explosive power tend to co‑occur.
- Body weight and body fat% are strongly correlated, as expected from body composition theory.
- Blood pressure correlates only weakly with performance metrics, suggesting it is more a health indicator than direct performance driver.


### 3.5 EDA Summary: Key Insights & Data Issues
Based on the Univariate, Bivariate, Categorical, and Correlation analyses conducted in sections §1.8 through §1.11, we can draw the following critical conclusions regarding the Body Performance dataset.

# 5 Key Insights
1. **Body Fat is a Prime Predictor:** `body fat_%` has the strongest negative correlation with performance. Individuals in Class 'A' consistently exhibit the lowest body fat percentages, while Class 'D' exhibits the highest.
2. **Strength & Agility Synergy:** Metrics like `broad jump_cm`, `sit-ups counts`, and `gripForce` move together. High scores in one usually indicate high scores in the others, and these are the primary drivers for achieving Class 'A' or 'B'.
3. **Gender Thresholds:** Both male and female distributions are relatively balanced across all four performance classes. This indicates that the grading system likely evaluates performance based on gender-specific physical thresholds.
4. **Age Distribution:** The dataset is slightly right-skewed regarding age, meaning the majority of our recorded individuals are younger. The models might be slightly biased toward younger physical performance baselines.
5. **Blood Pressure Irrelevance:** Systolic and diastolic blood pressures show very weak correlations with the physical performance metrics and the final class, suggesting they may not be strong standalone features for our predictive models.

### 3.6 Preprocessing Recommendations for the ML Team

1. Outliers in Target Metrics: The boxplots revealed distinct outliers in `gripForce` and `body fat_%` (e.g., unusually high body fat in Class B, or low grip force in Class A) that could skew sensitive machine learning models.

2. Multicollinearity:`weight_kg` and `height_cm` are highly correlated. Keeping both might affect the weights of linear models.

3. Differing Scales: Our features have vastly different scales (e.g., `body fat_%` is in the tens, while `broad jump_cm` is in the hundreds).

4. Categorical Variables:`gender` (M/F) and `class` (A/B/C/D) are currently in text/object format and cannot be natively processed by most ML algorithms.

5. Dimensionality:There are 11 independent features. While not massive, we need to ensure all features actually contribute to the model's accuracy to avoid the curse of dimensionality.

Note: I have already proactively applied IQR capping in Step 9.5 to ensure the correlation matrix was accurate.

To ensure our classification models achieve maximum accuracy, I recommend the following steps be taken in the Data Preparation phase:

Outlier Handling:Use the IQR (Interquartile Range) method to cap or remove the extreme outliers identified in the boxplots.

Feature Scaling:Apply `StandardScaler` or `MinMaxScaler` to all continuous numerical features so that distance-based models (like KNN or SVM) treat all metrics equally.

Encoding:Apply Binary Encoding to the `gender` column (e.g., M=1, F=0) and Ordinal Encoding to the target `class` column (A=0, B=1, C=2, D=3).

Feature Engineering (Optional):Consider combining `height_cm` and `weight_kg` into a single `BMI` feature to solve the multicollinearity issue and provide a unified health metric.


###  I recommend that the data preparation team uses the IQR (Interquartile Range) method to cap the extreme outliers found in the gripForce and body fat_% columns before training the machine learning models." You already have this covered in the EDA Summary I provided earlier! You are perfectly on track.

## **4. Machine Learning Model Development**

---

### **4.1 Feature Engineering & Data Preparation**
To prepare the data for Scikit-Learn algorithms, we encode categorical variables (Gender and Class) and scale the numerical features to ensure distance-based models perform optimally.

### 4.1 Feature Engineering and Modeling Dataset

In [ ]:
# ==================================================================
# 4.1.1 Encoding and Splitting
# ==================================================================

model_df = clean_df.copy()

# Binary Encoding for Gender
model_df['gender'] = model_df['gender'].map({'M': 0, 'F': 1})

# Ordinal Encoding for Class (Target)
class_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
model_df['class_encoded'] = model_df['class'].map(class_map)

# Feature/Target Separation
X = model_df.drop(columns=['class', 'class_encoded'])
y = model_df['class_encoded']

# 70/30 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"[INFO] Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")

## **4. Machine Learning Model Development**

---

This section implements the complete ML pipeline for both tasks:

| Task | Target Variable | Problem Type | Models |
|------|----------------|--------------|--------|
| **Classification** | `class` (A / B / C / D) | Multi-class | KNN · Logistic Regression · Decision Tree · Random Forest · SVM · Neural Network |
| **Regression** | `broad jump_cm` (continuous) | Regression | Linear Regression · Random Forest · SVR |

**Pipeline for each model:**
1. Data preparation & feature/target split
2. Train-Test splits: **80/20 · 70/30 · 50/50**
3. Feature scaling (StandardScaler)
4. Model training
5. Evaluation metrics (Accuracy, Precision, Recall, F1 / R², MAE, RMSE)
6. Confusion matrix / residual plots
7. K-Fold Cross-Validation (k=5)
8. Interpretation & insights

### **4.1 Shared Data Preparation**

Encode categorical variables and define feature/target matrices used by every model.

In [ ]:
# ==================================================================
# 4.1 — Encoding & Feature/Target Definition
# ==================================================================

model_df = clean_df.copy()

# Binary encode gender: Male=0, Female=1
model_df['gender'] = model_df['gender'].map({'M': 0, 'F': 1})

# Ordinal encode class: A=0, B=1, C=2, D=3
class_map     = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
inv_class_map = {v: k for k, v in class_map.items()}
model_df['class_encoded'] = model_df['class'].map(class_map)
CLASS_LABELS  = ['A', 'B', 'C', 'D']

# ── Classification features / target ──────────────────────────────
X_clf = model_df.drop(columns=['class', 'class_encoded'])
y_clf = model_df['class_encoded']

# ── Regression features / target (exclude class string & target) ──
X_reg = model_df.drop(columns=['class', 'class_encoded', 'broad jump_cm'])
y_reg = model_df['broad jump_cm']

print(f"Classification  →  X: {X_clf.shape}, y: {y_clf.shape}")
print(f"Regression      →  X: {X_reg.shape}, y: {y_reg.shape}")
print(f"Classification features: {list(X_clf.columns)}")

### **4.2 Train-Test Split Strategy**

We evaluate every model under three split ratios to understand how training data volume affects performance.

| Split | Train % | Test % | Use case |
|-------|---------|--------|----------|
| **80/20** | 80 | 20 | Standard baseline — more training data |
| **70/30** | 70 | 30 | Balanced — larger evaluation set |
| **50/50** | 50 | 50 | Stress test — limited training data |

A **helper function** handles splitting + scaling for any ratio, preventing data leakage (scaler is always fit only on train).

In [ ]:
# ==================================================================
# 4.2 — Split & Scale Helper (used by every model below)
# ==================================================================

SPLITS = {'80/20': 0.20, '70/30': 0.30, '50/50': 0.50}
SEED   = 42

def make_splits(X, y, stratify=False, seed=SEED):
    """Return dict of {split_label: (X_tr_sc, X_te_sc, y_tr, y_te)}."""
    result = {}
    for label, test_sz in SPLITS.items():
        strat = y if stratify else None
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=test_sz, random_state=seed, stratify=strat)
        sc = StandardScaler()
        X_tr_sc = sc.fit_transform(X_tr)
        X_te_sc  = sc.transform(X_te)
        result[label] = (X_tr_sc, X_te_sc, y_tr, y_te, sc)
    return result

def eval_clf(model, X_te, y_te):
    """Return classification metrics dict."""
    y_p = model.predict(X_te)
    return {
        'Accuracy' : round(accuracy_score(y_te, y_p), 4),
        'Precision': round(precision_score(y_te, y_p, average='weighted', zero_division=0), 4),
        'Recall'   : round(recall_score(y_te, y_p,    average='weighted', zero_division=0), 4),
        'F1-Score' : round(f1_score(y_te, y_p,        average='weighted', zero_division=0), 4),
    }, y_p

def eval_reg(model, X_te, y_te):
    """Return regression metrics dict."""
    y_p = model.predict(X_te)
    mse = mean_squared_error(y_te, y_p)
    return {
        'R²'  : round(r2_score(y_te, y_p), 4),
        'MAE' : round(mean_squared_error(y_te, y_p)**0.5 * 0, 3) + round(abs(y_te.values - y_p).mean(), 3),
        'MSE' : round(mse, 3),
        'RMSE': round(mse**0.5, 3),
    }, y_p

def plot_split_comparison(results_dict, metric, title, ylabel, ax, higher_better=True):
    """Bar chart: metric per split on one axis."""
    labels = list(results_dict.keys())
    vals   = [results_dict[s][metric] for s in labels]
    colors = ['#3498DB', '#2ECC71', '#E74C3C']
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', linewidth=0.8)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_ylabel(ylabel)
    best_idx = vals.index(max(vals)) if higher_better else vals.index(min(vals))
    for j, (bar, v) in enumerate(zip(bars, vals)):
        color = 'gold' if j == best_idx else 'black'
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                f'{v:.4f}', ha='center', fontsize=9, fontweight='bold', color=color)
    ax.set_ylim(0, max(vals)*1.15 if higher_better else min(vals)*0.85)

print("Helper functions ready: make_splits | eval_clf | eval_reg | plot_split_comparison")

---
## **Model 1 — K-Nearest Neighbors (KNN)**

### 🔍 Algorithm Overview
KNN is a **non-parametric, instance-based** learner. It makes predictions by finding the K most similar training samples (neighbours) and:
- **Classification:** majority vote among K neighbours
- **Regression:** mean of K neighbours' target values

### ⚙️ Key Hyperparameters
| Parameter | Value Used | Effect |
|-----------|-----------|--------|
| `n_neighbors` (K) | 11 | Higher K → smoother decision boundary, less overfitting |
| `weights` | `'distance'` | Closer neighbours vote more strongly |
| `metric` | `'minkowski'` (p=2 = Euclidean) | Distance measure between points |

### ✅ Strengths
- Simple to understand and implement
- No training phase (lazy learner)
- Naturally handles multi-class problems

### ❌ Limitations
- Slow prediction on large datasets (computes all distances)
- Sensitive to irrelevant/unscaled features → **requires StandardScaler**
- Performance drops sharply in high-dimensional spaces (curse of dimensionality)

#### **1.1 KNN — Classification (Predicting Performance Class A/B/C/D)**

In [ ]:
# ==================================================================
# Model 1.1 — KNN Classification
# ==================================================================

knn_clf_splits = make_splits(X_clf, y_clf, stratify=True)
knn_clf_results  = {}
knn_clf_preds    = {}
knn_clf_models   = {}

print("=" * 55)
print("  KNN CLASSIFICATION — Results per Split")
print("=" * 55)

for split_label, (X_tr, X_te, y_tr, y_te, _) in knn_clf_splits.items():
    model = KNeighborsClassifier(n_neighbors=11, weights='distance', metric='minkowski')
    model.fit(X_tr, y_tr)
    metrics, y_pred = eval_clf(model, X_te, y_te)
    knn_clf_results[split_label] = metrics
    knn_clf_preds[split_label]   = (y_te, y_pred)
    knn_clf_models[split_label]  = model
    print(f"  {split_label}  →  Acc={metrics['Accuracy']:.4f}  "
          f"P={metrics['Precision']:.4f}  R={metrics['Recall']:.4f}  F1={metrics['F1-Score']:.4f}")

print()

# ── Detailed classification report (70/30 split) ──────────────────
y_te_70, y_pred_70 = knn_clf_preds['70/30']
print("Detailed Report (70/30 split):")
print(classification_report(y_te_70, y_pred_70, target_names=CLASS_LABELS))

In [ ]:
# ==================================================================
# Model 1.1 — KNN Classification: Confusion Matrices + Split Comparison
# ==================================================================

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

# ── Row 1: Confusion matrices for each split ─────────────────────
cmaps = ['Blues', 'Greens', 'Oranges']
for j, (split_label, (y_te, y_pred)) in enumerate(knn_clf_preds.items()):
    cm = confusion_matrix(y_te, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmaps[j],
                xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=axes[0, j])
    axes[0, j].set_title(f'KNN Clf — {split_label}\nAcc={knn_clf_results[split_label]["Accuracy"]:.4f}',
                          fontweight='bold')
    axes[0, j].set_xlabel('Predicted'); axes[0, j].set_ylabel('Actual')

# ── Row 1, col 4: Metric comparison across splits ────────────────
x = range(3); labels = list(knn_clf_results.keys())
for metric, color in zip(['Accuracy','F1-Score'], ['#3498DB','#E74C3C']):
    vals = [knn_clf_results[s][metric] for s in labels]
    axes[0, 3].plot(labels, vals, marker='o', color=color, label=metric, lw=2, markersize=8)
axes[0, 3].set_title('KNN Clf — Metrics vs Split', fontweight='bold')
axes[0, 3].set_ylabel('Score'); axes[0, 3].set_ylim(0.3, 1.0)
axes[0, 3].legend(); axes[0, 3].grid(True, alpha=0.4)

# ── Row 2: Bar charts for each metric ────────────────────────────
for j, (metric, higher) in enumerate(zip(['Accuracy','Precision','Recall','F1-Score'],
                                          [True, True, True, True])):
    plot_split_comparison(
        {s: knn_clf_results[s] for s in knn_clf_results},
        metric, f'KNN Clf — {metric}', metric, axes[1, j], higher)

plt.suptitle('KNN Classification — Complete Evaluation Across All Splits',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ==================================================================
# Model 1.1 — KNN Classification: 5-Fold Cross-Validation
# ==================================================================

knn_clf_cv = KNeighborsClassifier(n_neighbors=11, weights='distance')
pipe_knn_c = Pipeline([('sc', StandardScaler()), ('clf', knn_clf_cv)])

cv_scores_knn_c = cross_validate(
    pipe_knn_c, X_clf, y_clf, cv=5,
    scoring=['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted'],
    n_jobs=-1
)

print("KNN CLASSIFICATION — 5-Fold Cross-Validation")
print("=" * 45)
for metric_key, label in [('test_accuracy','Accuracy'), ('test_precision_weighted','Precision'),
                           ('test_recall_weighted','Recall'), ('test_f1_weighted','F1-Score')]:
    scores = cv_scores_knn_c[metric_key]
    print(f"  {label:12s}: {scores.round(4)}  →  Mean={scores.mean():.4f}  Std=±{scores.std():.4f}")

#### **1.2 KNN — Regression (Predicting Broad Jump Distance)**

KNN Regression predicts a continuous value by averaging the target values of the K nearest training neighbours.

In [ ]:
# ==================================================================
# Model 1.2 — KNN Regression
# ==================================================================

from sklearn.neighbors import KNeighborsRegressor

knn_reg_splits  = make_splits(X_reg, y_reg, stratify=False)
knn_reg_results = {}
knn_reg_preds   = {}

print("=" * 55)
print("  KNN REGRESSION — Results per Split")
print("=" * 55)

for split_label, (X_tr, X_te, y_tr, y_te, _) in knn_reg_splits.items():
    model = KNeighborsRegressor(n_neighbors=11, weights='distance')
    model.fit(X_tr, y_tr)
    metrics, y_pred = eval_reg(model, X_te, y_te)
    knn_reg_results[split_label] = metrics
    knn_reg_preds[split_label]   = (y_te, y_pred)
    print(f"  {split_label}  →  R²={metrics['R²']:.4f}  MAE={metrics['MAE']:.2f}  RMSE={metrics['RMSE']:.2f}")

In [ ]:
# ==================================================================
# Model 1.2 — KNN Regression: Actual vs Predicted + Residuals + Split Comparison
# ==================================================================

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

split_colors = {'80/20':'#3498DB', '70/30':'#2ECC71', '50/50':'#E74C3C'}

for j, (split_label, (y_te, y_pred)) in enumerate(knn_reg_preds.items()):
    # Actual vs Predicted
    axes[0, j].scatter(y_te, y_pred, alpha=0.3, s=12, color=split_colors[split_label])
    mn, mx = y_te.min(), y_te.max()
    axes[0, j].plot([mn, mx], [mn, mx], 'k--', lw=1.5, label='Perfect fit')
    axes[0, j].set_title(f'KNN Reg — {split_label}\nR²={knn_reg_results[split_label]["R²"]:.4f}',
                          fontweight='bold')
    axes[0, j].set_xlabel('Actual (cm)'); axes[0, j].set_ylabel('Predicted (cm)')
    axes[0, j].legend(fontsize=8)

# Overlay all splits on one axis
for split_label, (y_te, y_pred) in knn_reg_preds.items():
    axes[0, 3].scatter(y_te, y_pred, alpha=0.2, s=8,
                       color=split_colors[split_label], label=split_label)
mn_all = min(v[0].min() for v in knn_reg_preds.values())
mx_all = max(v[0].max() for v in knn_reg_preds.values())
axes[0, 3].plot([mn_all, mx_all], [mn_all, mx_all], 'k--', lw=1.5)
axes[0, 3].set_title('KNN Reg — All Splits Overlay', fontweight='bold')
axes[0, 3].legend(); axes[0, 3].set_xlabel('Actual'); axes[0, 3].set_ylabel('Predicted')

# Metric bar charts
for j, (metric, higher) in enumerate(zip(['R²','MAE','MSE','RMSE'],
                                          [True, False, False, False])):
    plot_split_comparison(knn_reg_results, metric, f'KNN Reg — {metric}', metric, axes[1, j], higher)

plt.suptitle('KNN Regression — Complete Evaluation Across All Splits',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 📌 KNN — Summary & Insights

| Aspect | Observation |
|--------|-------------|
| **Classification** | Achieves moderate accuracy; best with `distance` weighting |
| **Regression** | Good R² but RMSE increases as training set shrinks (50/50) |
| **Split Sensitivity** | Performance noticeably drops from 80/20 → 50/50 due to fewer reference points |
| **Scaling** | Critical — without StandardScaler, Euclidean distance is dominated by large-scale features |
| **Best split** | 80/20 consistently yields highest accuracy and R² |

---
## **Model 2 — Logistic Regression**

### 🔍 Algorithm Overview
Despite its name, Logistic Regression is a **linear classification** model. It models the probability that a sample belongs to a class using the **logistic (sigmoid) function**:

$$P(y=k|X) = \frac{e^{w_k \cdot X}}{\sum_j e^{w_j \cdot X}}$$

For multi-class problems (4 classes here), it uses the **multinomial (softmax)** extension.

### ⚙️ Key Hyperparameters
| Parameter | Value | Description |
|-----------|-------|-------------|
| `multi_class` | `'multinomial'` | Softmax over all 4 classes simultaneously |
| `solver` | `'lbfgs'` | Quasi-Newton optimizer; efficient for multinomial |
| `max_iter` | `1000` | Max iterations for convergence |
| `C` | `1.0` | Regularisation strength (1/λ); larger C = less regularisation |

### ✅ Strengths
- Highly interpretable — coefficients show feature direction & magnitude
- Fast training and prediction
- Outputs calibrated class probabilities

### ❌ Limitations
- Assumes **linear** decision boundary — struggles with non-linear class separations
- Sensitive to multicollinearity between features
- Requires feature scaling for convergence

#### **2.1 Logistic Regression — Classification**

In [ ]:
# ==================================================================
# Model 2.1 — Logistic Regression Classification
# ==================================================================

lr_clf_splits  = make_splits(X_clf, y_clf, stratify=True)
lr_clf_results = {}
lr_clf_preds   = {}
lr_clf_models  = {}

print("=" * 55)
print("  LOGISTIC REGRESSION — Results per Split")
print("=" * 55)

for split_label, (X_tr, X_te, y_tr, y_te, _) in lr_clf_splits.items():
    model = LogisticRegression(multi_class='multinomial', solver='lbfgs',
                               max_iter=1000, C=1.0, random_state=SEED)
    model.fit(X_tr, y_tr)
    metrics, y_pred = eval_clf(model, X_te, y_te)
    lr_clf_results[split_label] = metrics
    lr_clf_preds[split_label]   = (y_te, y_pred)
    lr_clf_models[split_label]  = model
    print(f"  {split_label}  →  Acc={metrics['Accuracy']:.4f}  "
          f"P={metrics['Precision']:.4f}  R={metrics['Recall']:.4f}  F1={metrics['F1-Score']:.4f}")

print()
y_te_70, y_pred_70 = lr_clf_preds['70/30']
print("Detailed Report (70/30 split):")
print(classification_report(y_te_70, y_pred_70, target_names=CLASS_LABELS))

In [ ]:
# ==================================================================
# Model 2.1 — Logistic Regression: Confusion Matrices + Coefficients
# ==================================================================

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

for j, (split_label, (y_te, y_pred)) in enumerate(lr_clf_preds.items()):
    cm = confusion_matrix(y_te, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
                xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=axes[0, j])
    axes[0, j].set_title(f'LR Clf — {split_label}\nAcc={lr_clf_results[split_label]["Accuracy"]:.4f}',
                          fontweight='bold')
    axes[0, j].set_xlabel('Predicted'); axes[0, j].set_ylabel('Actual')

# Coefficient heatmap (70/30 model)
model_70 = lr_clf_models['70/30']
coef_df = pd.DataFrame(model_70.coef_, index=CLASS_LABELS, columns=X_clf.columns)
sns.heatmap(coef_df, cmap='coolwarm', center=0, annot=True, fmt='.2f',
            linewidths=0.4, ax=axes[0, 3])
axes[0, 3].set_title('LR Coefficients per Class\n(70/30 model)', fontweight='bold')

# Bar charts
for j, metric in enumerate(['Accuracy','Precision','Recall','F1-Score']):
    plot_split_comparison(lr_clf_results, metric, f'LR Clf — {metric}', metric, axes[1, j])

plt.suptitle('Logistic Regression Classification — Complete Evaluation Across All Splits',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ==================================================================
# Model 2.1 — Logistic Regression: 5-Fold Cross-Validation
# ==================================================================

pipe_lr_c = Pipeline([('sc', StandardScaler()),
                       ('clf', LogisticRegression(multi_class='multinomial',
                                                   solver='lbfgs', max_iter=1000,
                                                   random_state=SEED))])
cv_lr_c = cross_validate(pipe_lr_c, X_clf, y_clf, cv=5,
                          scoring=['accuracy','precision_weighted','recall_weighted','f1_weighted'],
                          n_jobs=-1)

print("LOGISTIC REGRESSION — 5-Fold Cross-Validation")
print("=" * 50)
for key, label in [('test_accuracy','Accuracy'), ('test_precision_weighted','Precision'),
                   ('test_recall_weighted','Recall'), ('test_f1_weighted','F1-Score')]:
    s = cv_lr_c[key]
    print(f"  {label:12s}: {s.round(4)}  →  Mean={s.mean():.4f}  Std=±{s.std():.4f}")

### 📌 Logistic Regression — Summary & Insights

| Aspect | Observation |
|--------|-------------|
| **Decision boundary** | Linear — classes B and C (which overlap in feature space) are hardest |
| **Coefficients** | Reveal which features push toward each class (A/B/C/D) |
| **Split stability** | Relatively stable across splits — linear models generalise well |
| **Limitation** | Cannot capture non-linear interactions between e.g. age and grip force |
| **Best use** | Baseline model; provides probability calibration for comparison |

---
## **Model 3 — Decision Tree**

### 🔍 Algorithm Overview
A Decision Tree recursively **partitions the feature space** using axis-aligned splits, choosing thresholds that maximise purity at each node. For classification, node purity is measured by **Gini Impurity** or **Entropy (Information Gain)**. For regression, it minimises **MSE** within each leaf.

**Example split:** `broad jump_cm ≤ 170` → left subtree predicts class D/C; right subtree predicts A/B

### ⚙️ Key Hyperparameters
| Parameter | Value | Effect |
|-----------|-------|--------|
| `max_depth` | 8 (clf) / 6 (reg) | Limits tree depth; prevents overfitting |
| `min_samples_leaf` | 10 | Minimum samples per leaf; smooths boundaries |
| `criterion` | `'gini'` / `'squared_error'` | Splitting metric |

### ✅ Strengths
- **Fully interpretable** — can be visualised as a tree diagram
- No feature scaling required
- Handles both numerical and categorical features naturally
- Captures non-linear interactions and thresholds

### ❌ Limitations
- Prone to **overfitting** when tree is deep
- High variance — small data changes → very different tree structure
- Generally outperformed by ensemble methods (Random Forest)

#### **3.1 Decision Tree — Classification**

In [ ]:
# ==================================================================
# Model 3.1 — Decision Tree Classification
# ==================================================================

dt_clf_splits  = make_splits(X_clf, y_clf, stratify=True)
dt_clf_results = {}
dt_clf_preds   = {}
dt_clf_models  = {}

print("=" * 55)
print("  DECISION TREE CLASSIFICATION — Results per Split")
print("=" * 55)

for split_label, (X_tr, X_te, y_tr, y_te, _) in dt_clf_splits.items():
    model = DecisionTreeClassifier(max_depth=8, min_samples_leaf=10,
                                   criterion='gini', random_state=SEED)
    model.fit(X_tr, y_tr)
    metrics, y_pred = eval_clf(model, X_te, y_te)
    dt_clf_results[split_label] = metrics
    dt_clf_preds[split_label]   = (y_te, y_pred)
    dt_clf_models[split_label]  = model
    print(f"  {split_label}  →  Acc={metrics['Accuracy']:.4f}  "
          f"P={metrics['Precision']:.4f}  R={metrics['Recall']:.4f}  F1={metrics['F1-Score']:.4f}")

print()
y_te_70, y_pred_70 = dt_clf_preds['70/30']
print("Detailed Report (70/30 split):")
print(classification_report(y_te_70, y_pred_70, target_names=CLASS_LABELS))

In [ ]:
# ==================================================================
# Model 3.1 — Decision Tree Clf: Confusion Matrices + Feature Importance
# ==================================================================

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

for j, (split_label, (y_te, y_pred)) in enumerate(dt_clf_preds.items()):
    cm = confusion_matrix(y_te, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=axes[0, j])
    axes[0, j].set_title(f'DT Clf — {split_label}\nAcc={dt_clf_results[split_label]["Accuracy"]:.4f}',
                          fontweight='bold')
    axes[0, j].set_xlabel('Predicted'); axes[0, j].set_ylabel('Actual')

# Feature importance (70/30 model)
model_70 = dt_clf_models['70/30']
fi = pd.Series(model_70.feature_importances_, index=X_clf.columns).sort_values(ascending=True)
fi.tail(10).plot(kind='barh', ax=axes[0, 3], color='#27AE60', edgecolor='white')
axes[0, 3].set_title('DT Clf — Feature Importances\n(70/30 model)', fontweight='bold')
axes[0, 3].set_xlabel('Importance Score')

for j, metric in enumerate(['Accuracy','Precision','Recall','F1-Score']):
    plot_split_comparison(dt_clf_results, metric, f'DT Clf — {metric}', metric, axes[1, j])

plt.suptitle('Decision Tree Classification — Complete Evaluation Across All Splits',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

#### **3.2 Decision Tree — Regression**

In [ ]:
# ==================================================================
# Model 3.2 — Decision Tree Regression
# ==================================================================

dt_reg_splits  = make_splits(X_reg, y_reg, stratify=False)
dt_reg_results = {}
dt_reg_preds   = {}

print("=" * 55)
print("  DECISION TREE REGRESSION — Results per Split")
print("=" * 55)

for split_label, (X_tr, X_te, y_tr, y_te, _) in dt_reg_splits.items():
    model = DecisionTreeRegressor(max_depth=6, min_samples_leaf=10,
                                  criterion='squared_error', random_state=SEED)
    model.fit(X_tr, y_tr)
    metrics, y_pred = eval_reg(model, X_te, y_te)
    dt_reg_results[split_label] = metrics
    dt_reg_preds[split_label]   = (y_te, y_pred)
    print(f"  {split_label}  →  R²={metrics['R²']:.4f}  MAE={metrics['MAE']:.2f}  RMSE={metrics['RMSE']:.2f}")

In [ ]:
# ==================================================================
# Model 3.2 — Decision Tree Regression: Plots
# ==================================================================

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
split_colors = {'80/20':'#3498DB', '70/30':'#2ECC71', '50/50':'#E74C3C'}

for j, (split_label, (y_te, y_pred)) in enumerate(dt_reg_preds.items()):
    axes[0, j].scatter(y_te, y_pred, alpha=0.3, s=12, color=split_colors[split_label])
    mn, mx = y_te.min(), y_te.max()
    axes[0, j].plot([mn, mx], [mn, mx], 'k--', lw=1.5)
    axes[0, j].set_title(f'DT Reg — {split_label}\nR²={dt_reg_results[split_label]["R²"]:.4f}', fontweight='bold')
    axes[0, j].set_xlabel('Actual (cm)'); axes[0, j].set_ylabel('Predicted (cm)')

# Residual distribution (70/30)
y_te_70, y_pred_70 = dt_reg_preds['70/30']
residuals = y_te_70.values - y_pred_70
axes[0, 3].hist(residuals, bins=30, color='#27AE60', edgecolor='white', alpha=0.85)
axes[0, 3].axvline(0, color='red', linestyle='--', lw=1.5)
axes[0, 3].set_title('DT Reg — Residuals (70/30)', fontweight='bold')
axes[0, 3].set_xlabel('Residual (cm)'); axes[0, 3].set_ylabel('Count')

for j, (metric, higher) in enumerate(zip(['R²','MAE','MSE','RMSE'], [True,False,False,False])):
    plot_split_comparison(dt_reg_results, metric, f'DT Reg — {metric}', metric, axes[1, j], higher)

plt.suptitle('Decision Tree Regression — Complete Evaluation Across All Splits',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ==================================================================
# Model 3 — Decision Tree: 5-Fold Cross-Validation (Both Tasks)
# ==================================================================

# Classification CV
pipe_dt_c = Pipeline([('sc', StandardScaler()),
                       ('clf', DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, random_state=SEED))])
cv_dt_c = cross_validate(pipe_dt_c, X_clf, y_clf, cv=5,
                          scoring=['accuracy','f1_weighted'], n_jobs=-1)

print("DECISION TREE CLASSIFICATION — 5-Fold CV")
print(f"  Accuracy : {cv_dt_c['test_accuracy'].round(4)}  →  Mean={cv_dt_c['test_accuracy'].mean():.4f}")
print(f"  F1-Score : {cv_dt_c['test_f1_weighted'].round(4)}  →  Mean={cv_dt_c['test_f1_weighted'].mean():.4f}")

print()

# Regression CV
pipe_dt_r = Pipeline([('sc', StandardScaler()),
                       ('reg', DecisionTreeRegressor(max_depth=6, min_samples_leaf=10, random_state=SEED))])
cv_dt_r = cross_validate(pipe_dt_r, X_reg, y_reg, cv=5, scoring='r2', n_jobs=-1)
print("DECISION TREE REGRESSION — 5-Fold CV")
print(f"  R²       : {cv_dt_r['test_score'].round(4)}  →  Mean={cv_dt_r['test_score'].mean():.4f}")

### 📌 Decision Tree — Summary & Insights

| Aspect | Observation |
|--------|-------------|
| **Interpretability** | Highest of all models — each decision rule is explicit |
| **Overfitting risk** | Controlled via `max_depth=8` and `min_samples_leaf=10` |
| **Classification** | Competitive accuracy; struggles with B↔C boundary (overlap) |
| **Regression** | Adequate R² but step-wise prediction creates quantisation error |
| **Split sensitivity** | Moderate — deeper trees overfit more on 50/50 split |
| **vs Random Forest** | RF is strictly better by averaging many trees (reduces variance) |

---
## **Model 4 — Random Forest**

### 🔍 Algorithm Overview
Random Forest is a **bagging ensemble** of Decision Trees. It trains `n_estimators` trees, each on a **bootstrap sample** of the data with a **random subset of features** at each split. Predictions are aggregated by:
- **Classification:** majority vote
- **Regression:** average

This process reduces **variance** without increasing **bias**, making it robust to overfitting.

### ⚙️ Key Hyperparameters
| Parameter | Value | Effect |
|-----------|-------|--------|
| `n_estimators` | 200 | Number of trees; more = better but slower |
| `max_depth` | `None` | Trees grown fully; overfitting controlled by ensemble |
| `max_features` | `'sqrt'` (clf) / `'log2'` (reg) | Feature subsetting per split |
| `min_samples_leaf` | 5 | Minimum samples per leaf |

### ✅ Strengths
- Best out-of-the-box accuracy in this dataset
- Provides reliable **feature importance** scores
- Robust to outliers and missing patterns
- No feature scaling strictly required (tree-based)

### ❌ Limitations
- Less interpretable than a single tree
- Slower than single models for prediction
- Large memory footprint with many trees

#### **4.1 Random Forest — Classification**

In [ ]:
# ==================================================================
# Model 4.1 — Random Forest Classification
# ==================================================================

rf_clf_splits  = make_splits(X_clf, y_clf, stratify=True)
rf_clf_results = {}
rf_clf_preds   = {}
rf_clf_models  = {}

print("=" * 55)
print("  RANDOM FOREST CLASSIFICATION — Results per Split")
print("=" * 55)

for split_label, (X_tr, X_te, y_tr, y_te, _) in rf_clf_splits.items():
    model = RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                   max_features='sqrt', random_state=SEED, n_jobs=-1)
    model.fit(X_tr, y_tr)
    metrics, y_pred = eval_clf(model, X_te, y_te)
    rf_clf_results[split_label] = metrics
    rf_clf_preds[split_label]   = (y_te, y_pred)
    rf_clf_models[split_label]  = model
    print(f"  {split_label}  →  Acc={metrics['Accuracy']:.4f}  "
          f"P={metrics['Precision']:.4f}  R={metrics['Recall']:.4f}  F1={metrics['F1-Score']:.4f}")

print()
y_te_70, y_pred_70 = rf_clf_preds['70/30']
print("Detailed Report (70/30 split):")
print(classification_report(y_te_70, y_pred_70, target_names=CLASS_LABELS))

In [ ]:
# ==================================================================
# Model 4.1 — Random Forest Clf: Confusion Matrices + Feature Importance
# ==================================================================

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

for j, (split_label, (y_te, y_pred)) in enumerate(rf_clf_preds.items()):
    cm = confusion_matrix(y_te, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
                xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=axes[0, j])
    axes[0, j].set_title(f'RF Clf — {split_label}\nAcc={rf_clf_results[split_label]["Accuracy"]:.4f}',
                          fontweight='bold')
    axes[0, j].set_xlabel('Predicted'); axes[0, j].set_ylabel('Actual')

# Feature importance — 70/30 model
rf_70 = rf_clf_models['70/30']
fi_rf = pd.Series(rf_70.feature_importances_, index=X_clf.columns).sort_values(ascending=True)
fi_rf.tail(10).plot(kind='barh', ax=axes[0, 3], color='#8E44AD', edgecolor='white')
axes[0, 3].set_title('RF Clf — Feature Importances\n(70/30 model)', fontweight='bold')
axes[0, 3].set_xlabel('Importance Score')

for j, metric in enumerate(['Accuracy','Precision','Recall','F1-Score']):
    plot_split_comparison(rf_clf_results, metric, f'RF Clf — {metric}', metric, axes[1, j])

plt.suptitle('Random Forest Classification — Complete Evaluation Across All Splits',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

#### **4.2 Random Forest — Regression**

In [ ]:
# ==================================================================
# Model 4.2 — Random Forest Regression
# ==================================================================

rf_reg_splits  = make_splits(X_reg, y_reg, stratify=False)
rf_reg_results = {}
rf_reg_preds   = {}
rf_reg_models  = {}

print("=" * 55)
print("  RANDOM FOREST REGRESSION — Results per Split")
print("=" * 55)

for split_label, (X_tr, X_te, y_tr, y_te, _) in rf_reg_splits.items():
    model = RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                  max_features='log2', random_state=SEED, n_jobs=-1)
    model.fit(X_tr, y_tr)
    metrics, y_pred = eval_reg(model, X_te, y_te)
    rf_reg_results[split_label] = metrics
    rf_reg_preds[split_label]   = (y_te, y_pred)
    rf_reg_models[split_label]  = model
    print(f"  {split_label}  →  R²={metrics['R²']:.4f}  MAE={metrics['MAE']:.2f}  RMSE={metrics['RMSE']:.2f}")

In [ ]:
# ==================================================================
# Model 4.2 — RF Regression: Actual vs Predicted + Residuals
# ==================================================================

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
split_colors = {'80/20':'#3498DB', '70/30':'#2ECC71', '50/50':'#E74C3C'}

for j, (split_label, (y_te, y_pred)) in enumerate(rf_reg_preds.items()):
    axes[0, j].scatter(y_te, y_pred, alpha=0.3, s=12, color=split_colors[split_label])
    mn, mx = y_te.min(), y_te.max()
    axes[0, j].plot([mn, mx], [mn, mx], 'k--', lw=1.5)
    axes[0, j].set_title(f'RF Reg — {split_label}\nR²={rf_reg_results[split_label]["R²"]:.4f}', fontweight='bold')
    axes[0, j].set_xlabel('Actual (cm)'); axes[0, j].set_ylabel('Predicted (cm)')

# Feature importance for regression
rf_reg_70 = rf_reg_models['70/30']
fi_rfr = pd.Series(rf_reg_70.feature_importances_, index=X_reg.columns).sort_values(ascending=True)
fi_rfr.plot(kind='barh', ax=axes[0, 3], color='#1D9E75', edgecolor='white')
axes[0, 3].set_title('RF Reg — Feature Importances\n(70/30 model)', fontweight='bold')

for j, (metric, higher) in enumerate(zip(['R²','MAE','MSE','RMSE'], [True,False,False,False])):
    plot_split_comparison(rf_reg_results, metric, f'RF Reg — {metric}', metric, axes[1, j], higher)

plt.suptitle('Random Forest Regression — Complete Evaluation Across All Splits',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ==================================================================
# Model 4 — Random Forest: 5-Fold Cross-Validation
# ==================================================================

pipe_rf_c = Pipeline([('sc', StandardScaler()),
                       ('clf', RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                                       random_state=SEED, n_jobs=-1))])
cv_rf_c = cross_validate(pipe_rf_c, X_clf, y_clf, cv=5,
                          scoring=['accuracy','precision_weighted','recall_weighted','f1_weighted'],
                          n_jobs=-1)

print("RANDOM FOREST CLASSIFICATION — 5-Fold CV")
print("=" * 50)
for key, label in [('test_accuracy','Accuracy'), ('test_precision_weighted','Precision'),
                   ('test_recall_weighted','Recall'), ('test_f1_weighted','F1-Score')]:
    s = cv_rf_c[key]
    print(f"  {label:12s}: {s.round(4)}  →  Mean={s.mean():.4f}  Std=±{s.std():.4f}")

pipe_rf_r = Pipeline([('sc', StandardScaler()),
                       ('reg', RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                                      random_state=SEED, n_jobs=-1))])
cv_rf_r = cross_validate(pipe_rf_r, X_reg, y_reg, cv=5, scoring='r2', n_jobs=-1)
print()
print("RANDOM FOREST REGRESSION — 5-Fold CV")
print(f"  R²: {cv_rf_r['test_score'].round(4)}  →  Mean={cv_rf_r['test_score'].mean():.4f}  Std=±{cv_rf_r['test_score'].std():.4f}")

### 📌 Random Forest — Summary & Insights

| Aspect | Observation |
|--------|-------------|
| **Classification accuracy** | Highest among all models; ensemble averaging removes tree variance |
| **Regression R²** | Excellent — captures non-linear interactions between physical features |
| **Feature importance** | `broad jump_cm`, `gripForce`, `sit-ups counts`, `body fat_%` are top predictors |
| **Split robustness** | Most stable across 80/20 → 50/50; ensemble compensates for fewer samples |
| **Overfitting control** | `min_samples_leaf=5` and feature subsampling prevent memorisation |
| **Recommended** | ✅ Best overall model for both classification and regression |

---
## **Model 5 — Support Vector Machine (SVM / SVR)**

### 🔍 Algorithm Overview
SVM finds the **maximum-margin hyperplane** that separates classes. For non-linearly separable data, it uses the **kernel trick** to implicitly map data into a higher-dimensional space where it becomes linearly separable.

**Key concept — the kernel:**
| Kernel | Formula | Use case |
|--------|---------|----------|
| `linear` | $K(x,z) = x \cdot z$ | Linearly separable data |
| `rbf` (Gaussian) | $K(x,z) = e^{-\gamma\|x-z\|^2}$ | Non-linear boundaries — **best here** |
| `poly` | $K(x,z) = (x \cdot z + 1)^d$ | Polynomial decision surfaces |

### ⚙️ Key Hyperparameters
| Parameter | Value | Effect |
|-----------|-------|--------|
| `C` | 10 | Regularisation — higher C = less margin, more accurate on train |
| `gamma` | `'scale'` | RBF bandwidth — auto-scaled to features |
| `kernel` | `'rbf'` | Handles non-linear class boundaries |
| `epsilon` (SVR) | 0.1 | Insensitive tube width for regression |

### ✅ Strengths
- Very effective in high-dimensional spaces
- Robust with proper C and γ tuning
- Strong theoretical guarantees (margin maximisation)

### ❌ Limitations
- **Does not scale well** to very large datasets (O(n²) memory)
- Requires careful hyperparameter tuning
- No direct probability output (needs Platt scaling)
- Feature scaling is **mandatory**

#### **5.1 SVM — Classification (Three Kernels + Tuned RBF)**

In [ ]:
# ==================================================================
# Model 5.1 — SVM Classification: Three Kernels
# ==================================================================

KERNELS = ['linear', 'rbf', 'poly']
kernel_results = {}

# Use 70/30 split for kernel comparison
_, (X_tr_70, X_te_70, y_tr_70, y_te_70, _) = list(
    make_splits(X_clf, y_clf, stratify=True).items())[1]

print("SVM CLASSIFICATION — Kernel Comparison (70/30 split)")
print("=" * 55)
for kernel in KERNELS:
    m = SVC(kernel=kernel, C=10, gamma='scale', random_state=SEED)
    m.fit(X_tr_70, y_tr_70)
    metrics, y_pred = eval_clf(m, X_te_70, y_te_70)
    kernel_results[kernel] = {'model': m, 'metrics': metrics, 'y_pred': y_pred}
    print(f"  {kernel:8s}  →  Acc={metrics['Accuracy']:.4f}  F1={metrics['F1-Score']:.4f}")

In [ ]:
# ==================================================================
# Model 5.1 — SVM Clf: All splits with best kernel (RBF) + Kernel comparison
# ==================================================================

svm_clf_splits  = make_splits(X_clf, y_clf, stratify=True)
svm_clf_results = {}
svm_clf_preds   = {}

print("=" * 55)
print("  SVM (RBF) CLASSIFICATION — Results per Split")
print("=" * 55)

for split_label, (X_tr, X_te, y_tr, y_te, _) in svm_clf_splits.items():
    model = SVC(kernel='rbf', C=10, gamma='scale', random_state=SEED)
    model.fit(X_tr, y_tr)
    metrics, y_pred = eval_clf(model, X_te, y_te)
    svm_clf_results[split_label] = metrics
    svm_clf_preds[split_label]   = (y_te, y_pred)
    print(f"  {split_label}  →  Acc={metrics['Accuracy']:.4f}  "
          f"P={metrics['Precision']:.4f}  R={metrics['Recall']:.4f}  F1={metrics['F1-Score']:.4f}")

print()
y_te_70b, y_pred_70b = svm_clf_preds['70/30']
print("Detailed Report (70/30, RBF kernel):")
print(classification_report(y_te_70b, y_pred_70b, target_names=CLASS_LABELS))

In [ ]:
# ==================================================================
# Model 5.1 — SVM: Kernel Comparison + Confusion Matrices + Split Bars
# ==================================================================

fig, axes = plt.subplots(3, 4, figsize=(22, 15))

# ── Row 0: Confusion matrices for 3 kernels (70/30) ──────────────
for j, kernel in enumerate(KERNELS):
    y_pred_k = kernel_results[kernel]['y_pred']
    cm = confusion_matrix(y_te_70, y_pred_k)
    acc = kernel_results[kernel]['metrics']['Accuracy']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=axes[0, j])
    axes[0, j].set_title(f'SVM {kernel} — 70/30\nAcc={acc:.4f}', fontweight='bold')
    axes[0, j].set_xlabel('Predicted'); axes[0, j].set_ylabel('Actual')

# Kernel bar chart
k_names = list(kernel_results.keys())
k_accs  = [kernel_results[k]['metrics']['Accuracy'] for k in k_names]
bars = axes[0, 3].bar(k_names, k_accs, color=['#3498DB','#E74C3C','#F39C12'], edgecolor='white')
axes[0, 3].set_title('SVM — Kernel Accuracy Comparison', fontweight='bold')
axes[0, 3].set_ylabel('Accuracy'); axes[0, 3].set_ylim(0.4, 1.0)
for bar, v in zip(bars, k_accs):
    axes[0, 3].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                    f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')

# ── Row 1: Confusion matrices for each split (RBF) ───────────────
for j, (split_label, (y_te, y_pred)) in enumerate(svm_clf_preds.items()):
    cm = confusion_matrix(y_te, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
                xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=axes[1, j])
    axes[1, j].set_title(f'SVM RBF — {split_label}\nAcc={svm_clf_results[split_label]["Accuracy"]:.4f}',
                          fontweight='bold')
    axes[1, j].set_xlabel('Predicted'); axes[1, j].set_ylabel('Actual')

# Accuracy line across splits
labels_svm = list(svm_clf_results.keys())
accs_svm   = [svm_clf_results[s]['Accuracy'] for s in labels_svm]
f1s_svm    = [svm_clf_results[s]['F1-Score'] for s in labels_svm]
axes[1, 3].plot(labels_svm, accs_svm, marker='o', color='#E74C3C', label='Accuracy', lw=2, markersize=8)
axes[1, 3].plot(labels_svm, f1s_svm,  marker='s', color='#3498DB', label='F1-Score', lw=2, markersize=8)
axes[1, 3].set_title('SVM RBF — Metrics vs Split', fontweight='bold')
axes[1, 3].set_ylabel('Score'); axes[1, 3].set_ylim(0.3, 1.0)
axes[1, 3].legend(); axes[1, 3].grid(True, alpha=0.4)

# ── Row 2: Metric bars ────────────────────────────────────────────
for j, metric in enumerate(['Accuracy','Precision','Recall','F1-Score']):
    plot_split_comparison(svm_clf_results, metric, f'SVM RBF — {metric}', metric, axes[2, j])

plt.suptitle('SVM Classification — Kernel Comparison + All Splits Evaluation',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

#### **5.2 SVM — Regression (SVR with RBF Kernel)**

In [ ]:
# ==================================================================
# Model 5.2 — SVR Regression
# ==================================================================

svr_reg_splits  = make_splits(X_reg, y_reg, stratify=False)
svr_reg_results = {}
svr_reg_preds   = {}

print("=" * 55)
print("  SVR REGRESSION — Results per Split")
print("=" * 55)

for split_label, (X_tr, X_te, y_tr, y_te, _) in svr_reg_splits.items():
    model = SVR(kernel='rbf', C=10, gamma='scale', epsilon=0.1)
    model.fit(X_tr, y_tr)
    metrics, y_pred = eval_reg(model, X_te, y_te)
    svr_reg_results[split_label] = metrics
    svr_reg_preds[split_label]   = (y_te, y_pred)
    print(f"  {split_label}  →  R²={metrics['R²']:.4f}  MAE={metrics['MAE']:.2f}  RMSE={metrics['RMSE']:.2f}")

In [ ]:
# ==================================================================
# Model 5.2 — SVR: Actual vs Predicted + Residuals + Bars
# ==================================================================

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
split_colors = {'80/20':'#3498DB', '70/30':'#2ECC71', '50/50':'#E74C3C'}

for j, (split_label, (y_te, y_pred)) in enumerate(svr_reg_preds.items()):
    axes[0, j].scatter(y_te, y_pred, alpha=0.3, s=12, color=split_colors[split_label])
    mn, mx = y_te.min(), y_te.max()
    axes[0, j].plot([mn, mx], [mn, mx], 'k--', lw=1.5)
    axes[0, j].set_title(f'SVR — {split_label}\nR²={svr_reg_results[split_label]["R²"]:.4f}', fontweight='bold')
    axes[0, j].set_xlabel('Actual (cm)'); axes[0, j].set_ylabel('Predicted (cm)')

y_te_70, y_pred_70 = svr_reg_preds['70/30']
residuals = y_te_70.values - y_pred_70
axes[0, 3].hist(residuals, bins=30, color='#E74C3C', edgecolor='white', alpha=0.85)
axes[0, 3].axvline(0, color='black', linestyle='--', lw=1.5)
axes[0, 3].set_title('SVR — Residual Distribution (70/30)', fontweight='bold')
axes[0, 3].set_xlabel('Residual (cm)'); axes[0, 3].set_ylabel('Count')

for j, (metric, higher) in enumerate(zip(['R²','MAE','MSE','RMSE'], [True,False,False,False])):
    plot_split_comparison(svr_reg_results, metric, f'SVR — {metric}', metric, axes[1, j], higher)

plt.suptitle('SVR Regression — Complete Evaluation Across All Splits',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ==================================================================
# Model 5 — SVM/SVR: 5-Fold Cross-Validation
# ==================================================================

pipe_svm_c = Pipeline([('sc', StandardScaler()),
                        ('clf', SVC(kernel='rbf', C=10, gamma='scale', random_state=SEED))])
cv_svm_c = cross_validate(pipe_svm_c, X_clf, y_clf, cv=5,
                           scoring=['accuracy','f1_weighted'], n_jobs=-1)

print("SVM CLASSIFICATION — 5-Fold CV")
print(f"  Accuracy : {cv_svm_c['test_accuracy'].round(4)}  →  Mean={cv_svm_c['test_accuracy'].mean():.4f}")
print(f"  F1-Score : {cv_svm_c['test_f1_weighted'].round(4)}  →  Mean={cv_svm_c['test_f1_weighted'].mean():.4f}")

pipe_svr = Pipeline([('sc', StandardScaler()),
                     ('reg', SVR(kernel='rbf', C=10, gamma='scale', epsilon=0.1))])
cv_svr = cross_validate(pipe_svr, X_reg, y_reg, cv=5, scoring='r2', n_jobs=-1)
print()
print("SVR REGRESSION — 5-Fold CV")
print(f"  R²       : {cv_svr['test_score'].round(4)}  →  Mean={cv_svr['test_score'].mean():.4f}")

### 📌 SVM — Summary & Insights

| Aspect | Observation |
|--------|-------------|
| **Best kernel** | RBF consistently outperforms linear and poly on this dataset |
| **Linear kernel** | Good baseline but misses non-linear class boundaries |
| **Classification** | Near top performance; very competitive with Random Forest |
| **SVR regression** | Strong R² with smooth predictions; no step-wise artefacts |
| **C sensitivity** | Higher C (=10) captures complex boundaries; cross-validate to confirm |
| **Scaling requirement** | Critical — SVM is distance-based; unscaled features break it |

---
## **Model 6 — Neural Network (Multi-Layer Perceptron)**

### 🔍 Algorithm Overview
An MLP is a **fully-connected feed-forward neural network**. Each neuron computes a weighted sum of its inputs, adds a bias, then passes the result through a non-linear **activation function**:

$$a = f(W \cdot x + b)$$

Multiple hidden layers allow the network to learn **hierarchical feature representations**, progressively combining simple patterns into complex ones.

**Architecture used:**
```
Input (11 features) → Dense(128, ReLU) → Dense(64, ReLU) → Output (4 classes / 1 value)
```

### ⚙️ Key Hyperparameters
| Parameter | Value | Effect |
|-----------|-------|--------|
| `hidden_layer_sizes` | `(128, 64)` | Two hidden layers; deeper → more capacity |
| `activation` | `'relu'` | ReLU avoids vanishing gradient; fast convergence |
| `solver` | `'adam'` | Adaptive momentum optimiser |
| `alpha` | `0.001` | L2 regularisation weight |
| `early_stopping` | `True` | Stops when validation loss stops improving |
| `max_iter` | `500` | Maximum epochs |

### ✅ Strengths
- Can model **highly non-linear** decision boundaries
- Early stopping prevents overfitting automatically
- Loss curve provides training transparency

### ❌ Limitations
- Least interpretable model
- Sensitive to hyperparameter choices
- Requires feature scaling; can be slow to train

#### **6.1 Neural Network — Classification**

In [ ]:
# ==================================================================
# Model 6.1 — MLP Classification
# ==================================================================

mlp_clf_splits  = make_splits(X_clf, y_clf, stratify=True)
mlp_clf_results = {}
mlp_clf_preds   = {}
mlp_clf_models  = {}

print("=" * 55)
print("  NEURAL NETWORK CLASSIFICATION — Results per Split")
print("=" * 55)

for split_label, (X_tr, X_te, y_tr, y_te, _) in mlp_clf_splits.items():
    model = MLPClassifier(
        hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
        alpha=0.001, max_iter=500, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=15, random_state=SEED
    )
    model.fit(X_tr, y_tr)
    metrics, y_pred = eval_clf(model, X_te, y_te)
    mlp_clf_results[split_label] = metrics
    mlp_clf_preds[split_label]   = (y_te, y_pred)
    mlp_clf_models[split_label]  = model
    print(f"  {split_label}  →  Acc={metrics['Accuracy']:.4f}  "
          f"P={metrics['Precision']:.4f}  R={metrics['Recall']:.4f}  F1={metrics['F1-Score']:.4f}  "
          f"[{model.n_iter_} epochs]")

print()
y_te_70, y_pred_70 = mlp_clf_preds['70/30']
print("Detailed Report (70/30 split):")
print(classification_report(y_te_70, y_pred_70, target_names=CLASS_LABELS))

In [ ]:
# ==================================================================
# Model 6.1 — MLP Clf: Confusion Matrices + Loss Curve + Split Bars
# ==================================================================

fig, axes = plt.subplots(3, 4, figsize=(22, 15))

# ── Row 0: Confusion matrices ─────────────────────────────────────
for j, (split_label, (y_te, y_pred)) in enumerate(mlp_clf_preds.items()):
    cm = confusion_matrix(y_te, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
                xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=axes[0, j])
    axes[0, j].set_title(f'MLP Clf — {split_label}\nAcc={mlp_clf_results[split_label]["Accuracy"]:.4f}',
                          fontweight='bold')
    axes[0, j].set_xlabel('Predicted'); axes[0, j].set_ylabel('Actual')

# Loss curve (70/30 model)
mlp_70 = mlp_clf_models['70/30']
axes[0, 3].plot(mlp_70.loss_curve_, color='#E74C3C', lw=2, label='Training loss')
if hasattr(mlp_70, 'validation_scores_') and mlp_70.validation_scores_ is not None:
    axes[0, 3].plot([1 - s for s in mlp_70.validation_scores_],
                    color='#3498DB', lw=2, linestyle='--', label='Val loss (1-acc)')
axes[0, 3].set_title('MLP — Training Loss Curve (70/30)', fontweight='bold')
axes[0, 3].set_xlabel('Epoch'); axes[0, 3].set_ylabel('Loss')
axes[0, 3].legend(); axes[0, 3].grid(True, alpha=0.4)

# ── Row 1: Metrics line + split comparison ─────────────────────
labels_nn = list(mlp_clf_results.keys())
axes[1, 0].plot(labels_nn, [mlp_clf_results[s]['Accuracy'] for s in labels_nn],
                marker='o', color='#E74C3C', lw=2, markersize=8, label='Accuracy')
axes[1, 0].plot(labels_nn, [mlp_clf_results[s]['F1-Score'] for s in labels_nn],
                marker='s', color='#3498DB', lw=2, markersize=8, label='F1-Score')
axes[1, 0].set_title('MLP — Metrics vs Split', fontweight='bold')
axes[1, 0].set_ylabel('Score'); axes[1, 0].set_ylim(0.3, 1.0)
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.4)

# Loss curve comparison across splits
for split_label, model in mlp_clf_models.items():
    axes[1, 1].plot(model.loss_curve_, label=split_label, lw=1.8)
axes[1, 1].set_title('MLP — Loss Curves All Splits', fontweight='bold')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.4)

# Bar charts
for j, metric in enumerate(['Accuracy','Precision','Recall','F1-Score']):
    plot_split_comparison(mlp_clf_results, metric, f'MLP Clf — {metric}', metric, axes[2, j])

# Hide unused axes[1, 2] and axes[1, 3]
axes[1, 2].axis('off'); axes[1, 3].axis('off')

plt.suptitle('Neural Network (MLP) Classification — Complete Evaluation Across All Splits',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

#### **6.2 Neural Network — Regression**

In [ ]:
# ==================================================================
# Model 6.2 — MLP Regression
# ==================================================================

from sklearn.neural_network import MLPRegressor

mlp_reg_splits  = make_splits(X_reg, y_reg, stratify=False)
mlp_reg_results = {}
mlp_reg_preds   = {}
mlp_reg_models  = {}

print("=" * 55)
print("  NEURAL NETWORK REGRESSION — Results per Split")
print("=" * 55)

for split_label, (X_tr, X_te, y_tr, y_te, _) in mlp_reg_splits.items():
    model = MLPRegressor(
        hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
        alpha=0.001, max_iter=500, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=15, random_state=SEED
    )
    model.fit(X_tr, y_tr)
    metrics, y_pred = eval_reg(model, X_te, y_te)
    mlp_reg_results[split_label] = metrics
    mlp_reg_preds[split_label]   = (y_te, y_pred)
    mlp_reg_models[split_label]  = model
    print(f"  {split_label}  →  R²={metrics['R²']:.4f}  MAE={metrics['MAE']:.2f}  RMSE={metrics['RMSE']:.2f}  [{model.n_iter_} epochs]")

In [ ]:
# ==================================================================
# Model 6.2 — MLP Regression: Actual vs Predicted + Loss Curves
# ==================================================================

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
split_colors = {'80/20':'#3498DB', '70/30':'#2ECC71', '50/50':'#E74C3C'}

for j, (split_label, (y_te, y_pred)) in enumerate(mlp_reg_preds.items()):
    axes[0, j].scatter(y_te, y_pred, alpha=0.3, s=12, color=split_colors[split_label])
    mn, mx = y_te.min(), y_te.max()
    axes[0, j].plot([mn, mx], [mn, mx], 'k--', lw=1.5)
    axes[0, j].set_title(f'MLP Reg — {split_label}\nR²={mlp_reg_results[split_label]["R²"]:.4f}', fontweight='bold')
    axes[0, j].set_xlabel('Actual (cm)'); axes[0, j].set_ylabel('Predicted (cm)')

# Loss curve comparison
for split_label, model in mlp_reg_models.items():
    axes[0, 3].plot(model.loss_curve_, label=split_label, lw=1.8)
axes[0, 3].set_title('MLP Reg — Loss Curves All Splits', fontweight='bold')
axes[0, 3].set_xlabel('Epoch'); axes[0, 3].set_ylabel('MSE Loss')
axes[0, 3].legend(); axes[0, 3].grid(True, alpha=0.4)

for j, (metric, higher) in enumerate(zip(['R²','MAE','MSE','RMSE'], [True,False,False,False])):
    plot_split_comparison(mlp_reg_results, metric, f'MLP Reg — {metric}', metric, axes[1, j], higher)

plt.suptitle('Neural Network (MLP) Regression — Complete Evaluation Across All Splits',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ==================================================================
# Model 6 — MLP: 5-Fold Cross-Validation
# ==================================================================

pipe_mlp_c = Pipeline([('sc', StandardScaler()),
                        ('clf', MLPClassifier(hidden_layer_sizes=(128,64), activation='relu',
                                              solver='adam', alpha=0.001, max_iter=300,
                                              early_stopping=True, random_state=SEED))])
cv_mlp_c = cross_validate(pipe_mlp_c, X_clf, y_clf, cv=5,
                           scoring=['accuracy','f1_weighted'], n_jobs=-1)

print("NEURAL NETWORK CLASSIFICATION — 5-Fold CV")
print(f"  Accuracy : {cv_mlp_c['test_accuracy'].round(4)}  →  Mean={cv_mlp_c['test_accuracy'].mean():.4f}")
print(f"  F1-Score : {cv_mlp_c['test_f1_weighted'].round(4)}  →  Mean={cv_mlp_c['test_f1_weighted'].mean():.4f}")

pipe_mlp_r = Pipeline([('sc', StandardScaler()),
                        ('reg', MLPRegressor(hidden_layer_sizes=(128,64), activation='relu',
                                             solver='adam', alpha=0.001, max_iter=300,
                                             early_stopping=True, random_state=SEED))])
cv_mlp_r = cross_validate(pipe_mlp_r, X_reg, y_reg, cv=5, scoring='r2', n_jobs=-1)
print()
print("NEURAL NETWORK REGRESSION — 5-Fold CV")
print(f"  R²       : {cv_mlp_r['test_score'].round(4)}  →  Mean={cv_mlp_r['test_score'].mean():.4f}")

### 📌 Neural Network — Summary & Insights

| Aspect | Observation |
|--------|-------------|
| **Classification** | Top-tier accuracy; rivals Random Forest with enough data |
| **Regression** | Captures non-linear relationships; competitive R² |
| **Loss curve** | Early stopping confirms the model stops before overfitting |
| **Split sensitivity** | More sensitive than RF on 50/50 — needs sufficient training data |
| **Epochs convergence** | 80/20 split converges faster; 50/50 may oscillate more |
| **Interpretability** | Lowest — treat as a black-box performance maximiser |

---
## **5. Global Model Comparison & Best Model Selection**

---

All six classification models and three regression models are compared side-by-side across every metric and split ratio.

In [ ]:
# ==================================================================
# 5.1 — Classification: Consolidate all results
# ==================================================================

ALL_CLF = {
    'KNN'               : knn_clf_results,
    'Logistic Regression': lr_clf_results,
    'Decision Tree'     : dt_clf_results,
    'Random Forest'     : rf_clf_results,
    'SVM (RBF)'         : svm_clf_results,
    'Neural Network'    : mlp_clf_results,
}

# Build comparison DataFrame
clf_rows = []
for model_name, split_dict in ALL_CLF.items():
    for split_label, metrics in split_dict.items():
        clf_rows.append({'Model': model_name, 'Split': split_label, **metrics})
clf_compare_df = pd.DataFrame(clf_rows)

print("=" * 70)
print("  CLASSIFICATION — ACCURACY PIVOT (Model × Split)")
print("=" * 70)
piv_acc = clf_compare_df.pivot(index='Model', columns='Split', values='Accuracy')[['80/20','70/30','50/50']]
print(piv_acc.round(4).to_string())

print()
print("=" * 70)
print("  CLASSIFICATION — F1-SCORE PIVOT")
print("=" * 70)
piv_f1 = clf_compare_df.pivot(index='Model', columns='Split', values='F1-Score')[['80/20','70/30','50/50']]
print(piv_f1.round(4).to_string())

In [ ]:
# ==================================================================
# 5.2 — Classification: Grouped Bar Charts + Heatmaps
# ==================================================================

model_order  = list(ALL_CLF.keys())
splits_order = ['80/20', '70/30', '50/50']
split_colors = {'80/20': '#3498DB', '70/30': '#2ECC71', '50/50': '#E74C3C'}
x = np.arange(len(model_order)); width = 0.25

fig, axes = plt.subplots(2, 2, figsize=(20, 12))

# ── Grouped bar: Accuracy ─────────────────────────────────────────
for j, split in enumerate(splits_order):
    vals = [ALL_CLF[m][split]['Accuracy'] for m in model_order]
    bars = axes[0,0].bar(x + j*width, vals, width, label=split,
                          color=split_colors[split], edgecolor='white', alpha=0.9)
    for bar, v in zip(bars, vals):
        axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                       f'{v:.3f}', ha='center', fontsize=7.5, rotation=90)
axes[0,0].set_xticks(x + width); axes[0,0].set_xticklabels(model_order, rotation=20, ha='right')
axes[0,0].set_ylim(0, 1.1); axes[0,0].set_title('Classification — Accuracy by Split', fontweight='bold')
axes[0,0].legend(title='Split'); axes[0,0].set_ylabel('Accuracy')

# ── Grouped bar: F1-Score ─────────────────────────────────────────
for j, split in enumerate(splits_order):
    vals = [ALL_CLF[m][split]['F1-Score'] for m in model_order]
    bars = axes[0,1].bar(x + j*width, vals, width, label=split,
                          color=split_colors[split], edgecolor='white', alpha=0.9)
    for bar, v in zip(bars, vals):
        axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                       f'{v:.3f}', ha='center', fontsize=7.5, rotation=90)
axes[0,1].set_xticks(x + width); axes[0,1].set_xticklabels(model_order, rotation=20, ha='right')
axes[0,1].set_ylim(0, 1.1); axes[0,1].set_title('Classification — F1-Score by Split', fontweight='bold')
axes[0,1].legend(title='Split'); axes[0,1].set_ylabel('F1-Score')

# ── Heatmap: Accuracy ─────────────────────────────────────────────
sns.heatmap(piv_acc.reindex(model_order), annot=True, fmt='.4f', cmap='YlGn',
            vmin=0.4, vmax=1.0, linewidths=0.5, ax=axes[1,0])
axes[1,0].set_title('Classification Accuracy Heatmap (Model × Split)', fontweight='bold')

# ── Heatmap: F1 ───────────────────────────────────────────────────
sns.heatmap(piv_f1.reindex(model_order), annot=True, fmt='.4f', cmap='YlGn',
            vmin=0.4, vmax=1.0, linewidths=0.5, ax=axes[1,1])
axes[1,1].set_title('Classification F1-Score Heatmap (Model × Split)', fontweight='bold')

plt.suptitle('All Models — Classification Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ==================================================================
# 5.3 — Regression: Consolidate all results
# ==================================================================

ALL_REG = {
    'KNN'            : knn_reg_results,
    'Decision Tree'  : dt_reg_results,
    'Random Forest'  : rf_reg_results,
    'SVR (RBF)'      : svr_reg_results,
    'Neural Network' : mlp_reg_results,
}

reg_rows = []
for model_name, split_dict in ALL_REG.items():
    for split_label, metrics in split_dict.items():
        reg_rows.append({'Model': model_name, 'Split': split_label, **metrics})
reg_compare_df = pd.DataFrame(reg_rows)

print("=" * 70)
print("  REGRESSION — R² PIVOT (Model × Split)")
print("=" * 70)
piv_r2 = reg_compare_df.pivot(index='Model', columns='Split', values='R²')[['80/20','70/30','50/50']]
print(piv_r2.round(4).to_string())

print()
print("=" * 70)
print("  REGRESSION — RMSE PIVOT")
print("=" * 70)
piv_rmse = reg_compare_df.pivot(index='Model', columns='Split', values='RMSE')[['80/20','70/30','50/50']]
print(piv_rmse.round(3).to_string())

In [ ]:
# ==================================================================
# 5.4 — Regression: Grouped Bar Charts + Heatmaps
# ==================================================================

reg_model_order = list(ALL_REG.keys())
x_reg = np.arange(len(reg_model_order))

fig, axes = plt.subplots(2, 2, figsize=(20, 12))

# ── R² ────────────────────────────────────────────────────────────
for j, split in enumerate(splits_order):
    vals = [ALL_REG[m][split]['R²'] for m in reg_model_order]
    bars = axes[0,0].bar(x_reg + j*width, vals, width, label=split,
                          color=split_colors[split], edgecolor='white', alpha=0.9)
    for bar, v in zip(bars, vals):
        axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                       f'{v:.3f}', ha='center', fontsize=8, rotation=90)
axes[0,0].set_xticks(x_reg + width); axes[0,0].set_xticklabels(reg_model_order, rotation=15, ha='right')
axes[0,0].set_ylim(0, 1.1); axes[0,0].set_title('Regression — R² by Split', fontweight='bold')
axes[0,0].legend(title='Split'); axes[0,0].set_ylabel('R²')

# ── RMSE ──────────────────────────────────────────────────────────
for j, split in enumerate(splits_order):
    vals = [ALL_REG[m][split]['RMSE'] for m in reg_model_order]
    bars = axes[0,1].bar(x_reg + j*width, vals, width, label=split,
                          color=split_colors[split], edgecolor='white', alpha=0.9)
    for bar, v in zip(bars, vals):
        axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                       f'{v:.2f}', ha='center', fontsize=8, rotation=90)
axes[0,1].set_xticks(x_reg + width); axes[0,1].set_xticklabels(reg_model_order, rotation=15, ha='right')
axes[0,1].set_title('Regression — RMSE by Split (lower = better)', fontweight='bold')
axes[0,1].legend(title='Split'); axes[0,1].set_ylabel('RMSE (cm)')

# ── Heatmaps ──────────────────────────────────────────────────────
sns.heatmap(piv_r2.reindex(reg_model_order), annot=True, fmt='.4f', cmap='Blues',
            vmin=0, vmax=1.0, linewidths=0.5, ax=axes[1,0])
axes[1,0].set_title('Regression R² Heatmap (Model × Split)', fontweight='bold')

sns.heatmap(piv_rmse.reindex(reg_model_order), annot=True, fmt='.2f', cmap='YlOrRd_r',
            linewidths=0.5, ax=axes[1,1])
axes[1,1].set_title('Regression RMSE Heatmap (lower = better)', fontweight='bold')

plt.suptitle('All Models — Regression Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ==================================================================
# 5.5 — Best Model per Task + Split (Summary Table)
# ==================================================================

print("=" * 65)
print("  BEST CLASSIFIER PER SPLIT (by Accuracy)")
print("=" * 65)
for split in splits_order:
    best_name = max(ALL_CLF, key=lambda m: ALL_CLF[m][split]['Accuracy'])
    best_m    = ALL_CLF[best_name][split]
    print(f"  {split}  →  {best_name:22s}  Acc={best_m['Accuracy']:.4f}  F1={best_m['F1-Score']:.4f}")

print()
print("=" * 65)
print("  BEST REGRESSOR PER SPLIT (by R²)")
print("=" * 65)
for split in splits_order:
    best_name = max(ALL_REG, key=lambda m: ALL_REG[m][split]['R²'])
    best_m    = ALL_REG[best_name][split]
    print(f"  {split}  →  {best_name:22s}  R²={best_m['R²']:.4f}  RMSE={best_m['RMSE']:.2f} cm")

print()
print("=" * 65)
print("  OVERALL RANKING — CLASSIFICATION (80/20, by Accuracy)")
print("=" * 65)
ranking = sorted(ALL_CLF.items(), key=lambda kv: kv[1]['80/20']['Accuracy'], reverse=True)
for rank, (name, splits) in enumerate(ranking, 1):
    m = splits['80/20']
    print(f"  #{rank}  {name:22s}  Acc={m['Accuracy']:.4f}  F1={m['F1-Score']:.4f}")

print()
print("=" * 65)
print("  OVERALL RANKING — REGRESSION (80/20, by R²)")
print("=" * 65)
ranking_r = sorted(ALL_REG.items(), key=lambda kv: kv[1]['80/20']['R²'], reverse=True)
for rank, (name, splits) in enumerate(ranking_r, 1):
    m = splits['80/20']
    print(f"  #{rank}  {name:22s}  R²={m['R²']:.4f}  RMSE={m['RMSE']:.2f} cm")

### **5.6 Feature Importance Analysis (Random Forest + Permutation)**

In [ ]:
# ==================================================================
# 5.6.1 — Random Forest Built-in Feature Importance
# ==================================================================

# Use 80/20 RF classifier
rf_80_clf = rf_clf_models['80/20']
fi_clf = pd.Series(rf_80_clf.feature_importances_, index=X_clf.columns).sort_values(ascending=False)

rf_80_reg = rf_reg_models['80/20']
fi_reg = pd.Series(rf_80_reg.feature_importances_, index=X_reg.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Classification importance
fi_clf.sort_values().plot(kind='barh', ax=axes[0], color='#8E44AD', edgecolor='white')
axes[0].set_title('RF Classification — Feature Importance (80/20)', fontweight='bold')
axes[0].set_xlabel('Importance Score')

# Regression importance
fi_reg.sort_values().plot(kind='barh', ax=axes[1], color='#1D9E75', edgecolor='white')
axes[1].set_title('RF Regression — Feature Importance (80/20)', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.suptitle('Random Forest Feature Importances', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Top 5 Classification Predictors:", fi_clf.head(5).index.tolist())
print("Top 5 Regression  Predictors:", fi_reg.head(5).index.tolist())

In [ ]:
# ==================================================================
# 5.6.2 — Permutation Feature Importance (Model-Agnostic)
# ==================================================================

# Build final RF pipeline on 80/20 split
X_tr_80, X_te_80, y_tr_80, y_te_80, sc_80 = rf_clf_splits['80/20']
rf_perm_pipe = Pipeline([('sc', StandardScaler()),
                          ('clf', RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))])
rf_perm_pipe.fit(X_tr_80, y_tr_80)

perm_result = permutation_importance(
    rf_perm_pipe, X_te_80, y_te_80,
    n_repeats=15, random_state=SEED, scoring='accuracy', n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature'   : X_clf.columns,
    'importance': perm_result.importances_mean,
    'std'       : perm_result.importances_std
}).sort_values('importance', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors_perm = ['#E74C3C' if v > 0 else '#BDC3C7' for v in perm_df['importance']]
bars = ax.barh(perm_df['feature'][::-1], perm_df['importance'][::-1],
               xerr=perm_df['std'][::-1], color=colors_perm[::-1],
               edgecolor='white', capsize=4)
ax.set_title('Permutation Feature Importance (Accuracy Drop when Shuffled)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Accuracy Drop (Δ Accuracy)')
ax.axvline(0, color='black', lw=0.8)
plt.tight_layout()
plt.show()

print(perm_df.to_string(index=False))

---
## **6. Final Conclusions & Recommendations**

---

### 🏆 Best Models

| Task | Best Model | Why |
|------|-----------|-----|
| **Classification** | **Random Forest** | Highest accuracy + F1, most stable across splits, interpretable feature importances |
| **Regression** | **Random Forest** | Best R², lowest RMSE, robust to outliers and non-linearity |

### 📊 Key Findings

1. **Body composition dominates** — `body fat_%` and `gripForce` are the strongest predictors of performance class
2. **Classes B and C are hardest to separate** — they share overlapping distributions across all features
3. **80/20 split is optimal** — all models perform best with more training data; 50/50 consistently degrades every model
4. **Random Forest is the most robust** — smallest performance drop from 80/20 → 50/50 among all models
5. **Linear models are insufficient** — Logistic Regression and KNN cannot capture the non-linear class boundaries
6. **SVM (RBF) is a strong alternative** — near-RF performance with stricter mathematical guarantees
7. **Neural Networks need data** — competitive on 80/20 but more sensitive on smaller splits

### 📏 Split Sensitivity Ranking (most → least robust)
`Random Forest ≈ SVM (RBF) > Neural Network > Decision Tree > Logistic Regression > KNN`

### 🔬 Recommendations for Future Work
- **Hyperparameter tuning** via GridSearchCV for every model
- **SMOTE** to handle any class imbalance that emerges in sub-population analysis  
- **Stacking ensemble** combining RF + SVM + MLP predictions
- **Deep learning** (Keras/PyTorch) for the regression task to push R² further
- **SHAP values** for global + local model explanation beyond permutation importance

## **6. Intelligent Deployment: Gradio Interface**

---

### **6.1 Real-time Inference Engine**
The final stage of our pipeline is the deployment of an interactive application. This tool allows users to input physiological data and receive an immediate classification of body performance based on our best-performing model (Random Forest).

In [ ]:
# 6.1.1 Application overview and setup
# Redundant imports removed as they are present in section 1.1


# Ensure X and y_class are consistent with the cleaned model_df
X = model_df.drop(columns=["class", "class_encoded"])
y_class = model_df["class_encoded"]

# 1) Train–test split (if not already done)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_class, test_size=0.3, random_state=42, stratify=y_class
)

scaler = StandardScaler()

# 2) Define and train final Random Forest classifier pipeline
clf_pipe = Pipeline([
    ("scaler", scaler),
    ("clf", RandomForestClassifier(n_estimators=200, random_state=42))
])

clf_pipe.fit(X_train, y_train)

In [ ]:
reg_pipe = Pipeline([
    ("scaler", scaler),
    ("reg", RandomForestRegressor(n_estimators=200, random_state=42))
])

reg_pipe.fit(X_train, y_reg.loc[y_train.index])

In [ ]:
# 6.1.2 Prepare mappings, feature order, and feature importance

# Fix: Use 'class_map' (the correct variable name from previous cells)
inv_class_mapping = {v: k for k, v in class_map.items()}
feature_order = list(X.columns)

rf_model = clf_pipe.named_steps["clf"]
rf_importances = rf_model.feature_importances_

feat_imp_df = (
    pd.DataFrame({"feature": feature_order, "importance": rf_importances})
      .sort_values(by="importance", ascending=False)
      .reset_index(drop=True)
)

### **6.2 Gradio Prediction Engine (Classification + Regression)**

In [ ]:
# Prediction function (used in the "Prediction" tab)

# Define clf_pipe and reg_pipe for the Gradio app
# Assuming rf_pipe (RandomForestClassifier pipeline) and reg_pipe (best regressor pipeline) are already defined and trained
# Based on prior cells, rf_pipe was defined in LgSxapSB-8aF, but reg_pipe was not explicitly defined.
# Let's create clf_pipe for the best classifier (Random Forest) and also a reg_pipe for the best regressor (Linear Regression).

# Re-using the scaler and models defined earlier
# The best classifier was RandomForestClassifier (stored in rf_perm_pipe or rf_pipe)
# The best regressor was Linear Regression (stored in reg_models["Linear Regression"])

# Assuming 'scaler' is already defined from previous cells (StandardScaler)
# Ensure X_train, y_train are available from previous splits

# Recreate the best classification pipeline for clf_pipe if needed, or use the existing rf_pipe
# From cell LgSxapSB-8aF, rf_pipe was created using X_train, y_train, and scaler.

# Re-create clf_pipe based on the best performing model (Random Forest) and the scaler
# Reusing objects 'scaler' from earlier executions
clf_pipe = Pipeline([
    ("scaler", StandardScaler()), # Use a new scaler instance for this pipeline or the one from before
    ("clf", RandomForestClassifier(n_estimators=200, random_state=42)) # Using the parameters of the best RF
])

# Fit the clf_pipe once (assuming X_train, y_train are available from previous cells)
# Ensure the scaler is fitted on the full training data before pipeline creation, or use a new one inside
clf_pipe.fit(X_train, y_train)

# Re-create reg_pipe for the best regressor (Linear Regression) for the Gradio app
reg_pipe = Pipeline([
    ("scaler", StandardScaler()), # Use a new scaler instance for this pipeline
    ("reg", LinearRegression()) # Best regressor identified
])

# *IMPORTANT*: Re-fit reg_pipe after X_for_regression definition was fixed
reg_pipe.fit(X_train_r, y_train_r) # Fit with regression training data


def predict_body_performance(
    age,
    gender,
    height_cm,
    weight_kg,
    body_fat,
    diastolic,
    systolic,
    gripForce,
    sit_and_bend,
    sit_ups,
    broad_jump
):
    """
    Take raw user inputs for one participant, apply the same encoding and
    feature ordering used during training, and return:
    - Predicted performance class (A/B/C/D)
    - Class probabilities as a DataFrame
    - Predicted broad jump distance (cm)
    """

    # Encode gender as in training (M -> 0, F -> 1)
    gender_num = 0 if gender == "M" else 1

    # Build single-row DataFrame with correct feature names
    data = {
        "age": age,
        "gender": gender_num,
        "height_cm": height_cm,
        "weight_kg": weight_kg,
        "body fat_%": body_fat,
        "diastolic": diastolic,
        "systolic": systolic,
        "gripForce": gripForce,
        "sit and bend forward_cm": sit_and_bend,
        "sit-ups counts": sit_ups,
        "broad jump_cm": broad_jump,
    }

    # For classification, 'broad jump_cm' is a feature. The problem is that the user input 'broad_jump' is for the target of regression.
    # The X dataframe for classification should not include the target variable for regression if it's meant to be predicted.
    # However, the original X dataframe (used for classification and regression) *does* contain 'broad jump_cm'.
    # This means the model is trained to predict 'class' using 'broad jump_cm' as a feature, which is okay.
    # But for the gradio prediction, if `broad_jump` is to be predicted, it should not be an input feature to the regressor.
    # It's also an input to the classifier which predicts 'class'. This is a bit conflicting.

    # Re-examine the original X definition:
    # X = model_df.drop(columns=["class", "class_encoded"]) -> X contains 'broad jump_cm'
    # y_class = model_df["class_encoded"]
    # y_reg = model_df["broad jump_cm"]

    # This implies the classification models use 'broad jump_cm' as a feature to predict 'class'.
    # And the regression models predict 'broad jump_cm' from the *same* X (which includes 'broad jump_cm'). This is incorrect for regression if broad_jump_cm is the target.

    # Let's correct the X for regression model training to exclude 'broad jump_cm' if it's the target.
    # I will modify the regression split (vjRWFdrO9QXO) to remove 'broad jump_cm' from X for regression.

    # Re-evaluate predict_body_performance function parameters for classification and regression.
    # For classification, all inputs are features.
    # For regression, all inputs *except* broad_jump are features.

    # Let's refine the input data construction for x_df to differentiate for regressor and classifier.
    x_classifier_df = pd.DataFrame([data]) # Initially has broad_jump_cm as a feature
    x_regressor_data = data.copy()
    x_regressor_data.pop("broad jump_cm") # Remove target for regression
    x_regressor_df = pd.DataFrame([x_regressor_data])

    # Ensure feature order for both
    classifier_feature_order = feature_order # This already includes 'broad jump_cm'
    regressor_feature_order = [f for f in feature_order if f != "broad jump_cm"]

    x_classifier_df = x_classifier_df[classifier_feature_order]
    x_regressor_df = x_regressor_df[regressor_feature_order]

    # Classification prediction
    class_pred_num = clf_pipe.predict(x_classifier_df)[0]
    class_proba = clf_pipe.predict_proba(x_classifier_df)[0]
    class_label = inv_class_mapping[class_pred_num]

    # Build probabilities DataFrame for plotting
    classes_ordered = [inv_class_mapping[i] for i in sorted(inv_class_mapping.keys())]
    proba_df = pd.DataFrame({
        "class": classes_ordered,
        "probability": class_proba
    })

    # Regression prediction
    broad_pred = reg_pipe.predict(x_regressor_df)[0]

    # Text summary
    summary_text = (
        f"Predicted performance class: {class_label}\n"
        f"Predicted broad jump: {broad_pred:.1f} cm"
    )

    return summary_text, proba_df

### 6.3 Interactive Input Controls (Participant Profile)

In [ ]:
#6.3.1 Helper function to plot feature importance (used in "Model Insights" tab)

def plot_feature_importance(top_n=10):
    """
    Return a Matplotlib figure with the top-N feature importances
    for the Random Forest classifier.
    """
    top_df = feat_imp_df.head(top_n).copy()

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.barplot(
        data=top_df,
        x="importance",
        y="feature",
        hue="feature", # Assign y variable to hue
        palette="Blues_r",
        legend=False, # Set legend to False
        ax=ax
    )
    ax.set_title("Top Feature Importances (Random Forest)")
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")
    fig.tight_layout()
    return fig

In [ ]:
#6.3.2 Build the Gradio interface with tabs

with gr.Blocks() as demo:
    gr.Markdown("## Body Performance AI – Final Project Demo")
    gr.Markdown(
        "This interactive app is powered by the **final trained models in this notebook**. "
        "Use the controls to simulate a participant and view the predicted performance class, "
        "probabilities, and broad jump distance."
    )

    with gr.Tab("Prediction"):
        gr.Markdown("### Enter Participant Measurements")

        with gr.Row():
            with gr.Column():
                age = gr.Slider(18, 80, value=30, step=1, label="Age")
                gender = gr.Radio(["M", "F"], value="M", label="Gender")
                height_cm = gr.Slider(140.0, 200.0, value=170.0, step=0.1, label="Height (cm)")
                weight_kg = gr.Slider(40.0, 130.0, value=70.0, step=0.1, label="Weight (kg)")
                body_fat = gr.Slider(3.0, 45.0, value=22.0, step=0.1, label="Body fat (%)")

            with gr.Column():
                diastolic = gr.Slider(40.0, 130.0, value=80.0, step=1.0, label="Diastolic BP (mmHg)")
                systolic = gr.Slider(70.0, 200.0, value=130.0, step=1.0, label="Systolic BP (mmHg)")
                gripForce = gr.Slider(5.0, 70.0, value=35.0, step=0.5, label="Grip Force")
                sit_and_bend = gr.Slider(-5.0, 60.0, value=15.0, step=0.5, label="Sit and Bend Forward (cm)")
                sit_ups = gr.Slider(0.0, 80.0, value=40.0, step=1.0, label="Sit-ups counts")
                broad_jump = gr.Slider(80.0, 300.0, value=190.0, step=1.0, label="Broad jump (cm)")

        predict_btn = gr.Button("Predict")

        summary_output = gr.Textbox(
            label="Prediction Summary",
            lines=3,
            interactive=False
        )

        proba_plot = gr.BarPlot(
            label="Class Probabilities",
            x="class",
            y="probability",
            x_title="Class",
            y_title="Probability",
            width=500,
            height=350
        )

        predict_btn.click(
            fn=predict_body_performance,
            inputs=[
                age, gender, height_cm, weight_kg, body_fat,
                diastolic, systolic, gripForce, sit_and_bend, sit_ups, broad_jump
            ],
            outputs=[summary_output, proba_plot]
        )

    with gr.Tab("Model Insights"):
        gr.Markdown("### Random Forest Feature Importance")

        top_n_slider = gr.Slider(
            minimum=3,
            maximum=len(feature_order),
            value=10,
            step=1,
            label="Number of top features to display"
        )

        def _plot_importance_wrapper(n):
            return plot_feature_importance(top_n=int(n))

        # Initialize with default top 10 by passing the initial value
        fig_output = gr.Plot(
            label="Top Feature Importances",
            value=_plot_importance_wrapper(top_n_slider.value)
        )

        top_n_slider.change(
            fn=_plot_importance_wrapper,
            inputs=top_n_slider,
            outputs=fig_output
        )

demo.launch()

> The demo above is directly connected to the Random Forest classifier and regression models trained in the previous sections.  
> Any change in the training code (e.g., different preprocessing or hyperparameters) will automatically be reflected in the app after re‑running the notebook.